In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Optional

## LAYER 0 — SIGNAL DETECTION (CUSUM / EWMA)

In [7]:
# LAYER 0— SIGNAL DETECTION (CUSUM / EWMA)
SEED= 42
usage= pd.read_csv('Usage3.csv')
usage= usage[usage['AtHomePatientId'].notna()].copy()
usage['AtHomePatientId']= usage['AtHomePatientId'].astype(int)
usage['ReferenceDate']= pd.to_datetime(
    usage['ReferenceDate'].fillna(0).astype(int).astype(str),
    format='%Y%m%d', errors='coerce')
usage= usage[
    usage['ReferenceDate'].notna() &
    (usage['ReferenceDate'] >= '2020-01-01') &
    (usage['ReferenceDate'] <= '2030-12-31')].sort_values(['AtHomePatientId','ReferenceDate']).reset_index(drop=True)

In= pd.read_csv('Intervention3.csv', engine='python', on_bad_lines='skip')
In= In.rename(columns={'s': 'Status'})
In['AtHomePatientId'] = In['AtHomePatientId'].astype(int)
In['ReferenceDate'] = pd.to_datetime(
    In['ReferenceDate'].astype(str).str.split('.').str[0], errors='coerce')
In= In[In['ReferenceDate'].notna()].reset_index(drop=True)

bio_mapping = pd.read_csv('biomarker_enrollment_mapping.csv')
ww = pd.read_csv('db_withings_watch_connected.csv')
bio_mapping['AtHomePatientId'] = bio_mapping['AtHomePatientId'].astype(int)
ww['AtHomePatientId'] = ww['AtHomePatientId'].astype(int)
ww['timestamp'] = pd.to_datetime(ww['timestamp'], errors='coerce', utc=True)
poc_ids = set(bio_mapping['AtHomePatientId'])

#VECTORIZED LAYER 0 — BASELINE

usage['rec_n']= usage.groupby('AtHomePatientId').cumcount()

bl= (usage[usage['rec_n'] < 56]
    .groupby('AtHomePatientId')
    .agg(use_mean= ('Use','mean'),use_std= ('Use','std'),
        ahi_mean= ('AHI','mean'),ahi_std= ('AHI','std'),leaks95_mean= ('Leaks95', 'mean'),n_records= ('Use','count'),).reset_index())
bl['use_std']= bl['use_std'].fillna(0.1).clip(lower=0.1)
bl['ahi_std']= bl['ahi_std'].fillna(0.1).clip(lower=0.1)
bl['k_use']= 0.5 * bl['use_std'];  bl['h_use'] = 4.0 * bl['use_std']
bl['k_ahi']= 0.5 * bl['ahi_std'];  bl['h_ahi'] = 4.0 * bl['ahi_std']
bl= bl.merge(usage.groupby('AtHomePatientId')['ReferenceDate'].max().reset_index(name='last_date'),on='AtHomePatientId')
bl= bl[bl['n_records'] >= 14].reset_index(drop=True)

#VECTORIZED CUSUM / EWMA

u= usage[['AtHomePatientId', 'ReferenceDate', 'Use', 'AHI']].merge(
    bl[['AtHomePatientId', 'use_mean', 'use_std', 'ahi_mean', 'ahi_std',
        'k_use', 'h_use', 'k_ahi', 'h_ahi', 'last_date']],
    on='AtHomePatientId', how='inner')
u['inc_use']= -(u['Use'] - u['use_mean'] + u['k_use'])
u['inc_ahi']=  (u['AHI'] - u['ahi_mean'] - u['k_ahi'])
u = u.drop(columns=['use_mean','use_std','ahi_mean','ahi_std','k_use','k_ahi'])

grp = u.groupby('AtHomePatientId')
u['cum_use']= grp['inc_use'].cumsum()
u['cum_ahi']= grp['inc_ahi'].cumsum()
u['S_neg']= (u['cum_use'] - grp['cum_use'].cummin()).clip(lower=0)
u['S_pos']= (u['cum_ahi'] - grp['cum_ahi'].cummin()).clip(lower=0)
u['ewma_z']= grp['Use'].transform(lambda s: s.ewm(alpha=0.2, adjust=False).mean())

layer0= grp.last()[['S_neg', 'S_pos', 'h_use', 'h_ahi', 'ewma_z', 'last_date']].reset_index()
layer0= layer0.merge(bl[['AtHomePatientId', 'use_mean', 'use_std', 'ahi_mean', 'ahi_std']],on='AtHomePatientId')
layer0['ewma_lcl']= layer0['use_mean'] - 3.0 * layer0['use_std'] * np.sqrt(0.2 / 1.8)
layer0['cusum_use_alarm']= layer0['S_neg'] > layer0['h_use']
layer0['cusum_ahi_alarm']= layer0['S_pos'] > layer0['h_ahi']
layer0['ewma_use_alarm']= layer0['ewma_z'] < layer0['ewma_lcl']
layer0['any_alarm']= (layer0['cusum_use_alarm'] | layer0['cusum_ahi_alarm'] | layer0['ewma_use_alarm'])
layer0['in_poc']= layer0['AtHomePatientId'].isin(poc_ids)

#14-DAY CPAP STABILITY

u14= u[(u['last_date']-u['ReferenceDate']).dt.days <= 14]

s14 = (u14.groupby('AtHomePatientId').agg(use14_m=('Use', 'mean'), use14_s=('Use', 'std'), ahi14_m=('AHI', 'mean')).reset_index())
s14 = s14.merge(bl[['AtHomePatientId','use_mean', 'ahi_mean', 'ahi_std']], on='AtHomePatientId')

s14['cv'] = (s14['use14_s'].fillna(0) / (s14['use14_m'] + 1e-6)).clip(0, 1)
s14['ahi_z'] = (abs(s14['ahi14_m'] - s14['ahi_mean'])/(s14['ahi_std'] + 1e-6) / 3).clip(0, 1)

slope_14d = (u14.sort_values('ReferenceDate').groupby('AtHomePatientId')['Use'].apply(lambda s: float(np.polyfit(range(len(s)), s.values, 1)[0]) if len(s) >= 3 else 0.0
    ).reset_index(name='use14_slope'))
s14 = s14.merge(slope_14d, on='AtHomePatientId', how='left')
s14['use14_slope'] = s14['use14_slope'].fillna(0.0)
s14['slope_penalty'] = (s14['use14_slope'].abs() / 0.5).clip(0, 1)

s14['sigma_cpap'] = (0.4 *(1-s14['cv'])+ 0.4* (1-s14['slope_penalty'])+ 0.2 * (1 - s14['ahi_z']))

#CARE-PROCESS STABILITY FROM PENDING INTERVENTIONS

pend= In[In['Status'] == 'Pending'].copy()
pend= pend.merge(layer0[['AtHomePatientId', 'last_date']], on='AtHomePatientId', how='left')
pend['age_days'] =(pend['last_date'] - pend['ReferenceDate']).dt.days.clip(lower=0)

oldest= pend.groupby('AtHomePatientId')['age_days'].max().reset_index(name='oldest_pend')
s14= s14.merge(oldest, on='AtHomePatientId', how='left')
s14['oldest_pend']= s14['oldest_pend'].fillna(0)
s14['sigma_care']= np.exp(-s14['oldest_pend'] / 14.0)

s14['sigma']=(0.60 * s14['sigma_cpap'] + 0.40 * s14['sigma_care']).clip(0, 1)

s14['window_days']=(pd.cut(s14['sigma'],bins = [-0.001, 0.30, 0.50, 0.70, 0.85, 1.001],labels= [4, 7, 14, 21, 28]).cat.codes.map({0: 4, 1: 7, 2: 14, 3: 21, 4: 28}).fillna(7)
    .astype(int))

layer0 = layer0.merge(s14[['AtHomePatientId', 'sigma', 'window_days']], on='AtHomePatientId', how='left')
n_pend = pend.groupby('AtHomePatientId').size().reset_index(name='n_pending')
layer0 = layer0.merge(n_pend, on='AtHomePatientId',how='left')
layer0['n_pending'] = layer0['n_pending'].fillna(0).astype(int)

#PoC PATIENTS: UPDATE SIGMA WITH WITHINGS BIOMARKER SIGNAL

ww_last = ww.copy()
ww_last['ts_tz'] = ww_last['timestamp'].dt.tz_localize(None)
ww_last =ww_last.merge(layer0[['AtHomePatientId', 'last_date']], on='AtHomePatientId', how='inner')
ww_7d = ww_last[(ww_last['last_date'] - ww_last['ts_tz']).dt.days <= 7]

bio_agg = ww_7d.groupby('AtHomePatientId').agg(spo2_mean = ('spo2','mean'),hrv_rmssd_mean = ('hrv_rmssd','mean'),
                                               sleep_score_mean= ('sleep_score','mean'),).reset_index()
bio_agg['sigma_bio']=((bio_agg['spo2_mean'] - 88).clip(0, 9) / 9 * 0.4+ bio_agg['hrv_rmssd_mean'].clip(0, 35) / 35 * 0.3
    + (bio_agg['sleep_score_mean']-40).clip(0, 55) / 55 * 0.3).clip(0, 1)

layer0 =layer0.merge(bio_agg[['AtHomePatientId','sigma_bio']], on='AtHomePatientId', how='left')
has_bio=layer0['sigma_bio'].notna()
layer0.loc[has_bio, 'sigma'] = ((layer0.loc[has_bio,'sigma'] * 0.90 + layer0.loc[has_bio, 'sigma_bio'] * 0.10).clip(0, 1))
layer0 = layer0.drop(columns=['sigma_bio'])

print(f'Layer 0:{len(layer0):,} patients')
print(f'\nAlarm summary:')
print(f'CUSUM use↓:{layer0["cusum_use_alarm"].sum():,}')
print(f'CUSUM AHI↑:{layer0["cusum_ahi_alarm"].sum():,}')
print(f'EWMA use↓: {layer0["ewma_use_alarm"].sum():,}')
print(f'Any alarm: {layer0["any_alarm"].sum():,}  ({layer0["any_alarm"].mean()*100:.1f}%)')

print(f'\nStability windows:')
for w in [4, 7, 14, 21, 28]:
    n=(layer0['window_days'] == w).sum()
    print(f'{w:>2}d : {n:,}')

print(f'\nTop 10 most unstable:')
print(layer0.sort_values('sigma').head(10)[['AtHomePatientId','use_mean','cusum_use_alarm',
         'ewma_use_alarm', 'S_neg','sigma', 'window_days','in_poc']].to_string(index=False))

layer0.to_csv('layer0_results.csv', index=False)

Layer 0:41,115 patients

Alarm summary:
CUSUM use↓:5,938
CUSUM AHI↑:7,587
EWMA use↓: 2,628
Any alarm: 12,111  (29.5%)

Stability windows:
 4d : 61
 7d : 1,261
14d : 3,379
21d : 8,344
28d : 28,070

Top 10 most unstable:
 AtHomePatientId  use_mean  cusum_use_alarm  ewma_use_alarm     S_neg    sigma  window_days  in_poc
           85892  6.880714             True           False 93.007983 0.043693            4   False
           80355  5.177679             True           False 34.467343 0.077022            4   False
          199007  0.242321            False           False  0.000000 0.107316            4   False
          106541  6.137143             True           False 14.316269 0.122685            4   False
           68989  3.335000            False           False  9.100726 0.137722            4   False
           21442  4.818077            False           False  2.281687 0.142220            4   False
          105026  5.731786            False           False  5.281606 0.161456   

## STEP 2 — DATA LOADING

In [10]:
#INTERVENTION DEFINITION
int_def= pd.read_csv("Interventiondefinition.csv")
int_def["Category"]= int_def["Category"].str.strip().str.title()
int_def["Channel"]= (int_def["Category"].map({"Visit": "Visit", "Call": "Call", "Sms": "SMS"}).fillna("Unknown"))

In= In.drop(columns=[c for c in ["Category", "Channel"] if c in In.columns])

In= pd.merge(In,int_def[["JobTypeCode", "Category", "Channel"]],on="JobTypeCode",how="left",)
In["Category"]= In["Category"].fillna("Unknown")
In["Channel"]= In["Channel"].fillna("Unknown")

#MONITORING
Su= pd.read_csv("Monitoring3.csv",engine = "python",on_bad_lines = "skip",quotechar = '"',)
Su["AtHomePatientId"] = pd.to_numeric(Su["AtHomePatientId"], errors="coerce")
Su= Su[Su["AtHomePatientId"].notna()].copy()
Su["AtHomePatientId"]= Su["AtHomePatientId"].astype(int)
Su["ExecutionDate"]= pd.to_datetime(Su["ExecutionDate"], errors="coerce")
Su["AnswerValue_num"]= pd.to_numeric(Su["AnswerValue"], errors="coerce")

Su["QuestionnaireId_num"]= pd.to_numeric(Su["QuestionnaireId"], errors="coerce")
su_ess= Su[(Su["QuestionnaireId_num"] == 267) &(Su["QuestionId"].astype(str).str.strip().str.strip("'") == "2208") &
    Su["AnswerValue_num"].notna() & Su["AnswerValue_num"].between(0, 24)].copy()
su_ess["survey_name"] = "ESS"
su_ess= su_ess.rename(columns={
    "ExecutionDate": "execution_date",
    "AnswerValue_num":"score_value",
})[["AtHomePatientId","execution_date", "survey_name","score_value"]].dropna()

print(f"ESS from Monitoring3:{len(su_ess):,} rows | "
      f"{su_ess['AtHomePatientId'].nunique():,} patients")

#CONNECTED MEDICAL SURVEYS (PoC cohort)

surv_med= pd.read_csv("db_surveys_medical_connected.csv")
surv_med["AtHomePatientId"]= surv_med["AtHomePatientId"].astype(int)
surv_med["execution_date"]= pd.to_datetime(surv_med["execution_date"], errors="coerce")

#main-cohort risk factor scores
rf= pd.read_csv("RF.csv")
rf["AtHomePatientId"]= rf["AtHomePatientId"].astype(int)
rf["CollectionDate"]= pd.to_datetime(rf["CollectionDate"], errors="coerce")

rf_ess= rf[(rf["RiskFactorId"] == 5) &
    pd.to_numeric(rf["RiskFactorValue"], errors="coerce").between(0, 24)].copy()
rf_ess["score_value"]= pd.to_numeric(rf_ess["RiskFactorValue"], errors="coerce")
rf_ess["survey_name"]= "ESS"
rf_ess = rf_ess.rename(columns={"CollectionDate": "execution_date"})[["AtHomePatientId", "execution_date", "survey_name", "score_value"]
].dropna()

print(f"ESS from RF.csv:{len(rf_ess):,} rows | "f"{rf_ess['AtHomePatientId'].nunique():,} patients")

surv_all= (pd.concat([surv_med[["AtHomePatientId", "execution_date", "survey_name", "score_value"]],
            su_ess,rf_ess,],ignore_index=True,)
    .dropna(subset=["AtHomePatientId", "execution_date", "survey_name", "score_value"])
    .sort_values(["AtHomePatientId", "survey_name", "execution_date"])
    .drop_duplicates(subset=["AtHomePatientId", "survey_name", "execution_date"], keep="last")
    .reset_index(drop=True))

print(f"\nsurv_all combined:{len(surv_all):,} rows")
print("ESS coverage: "f"{surv_all[surv_all['survey_name']=='ESS']['AtHomePatientId'].nunique():,} patients "
      f"(was 400 before fix)")

#BIOMARKER ENROLLMENT

bio_mapping= pd.read_csv("biomarker_enrollment_mapping.csv")
bio_mapping["AtHomePatientId"]= bio_mapping["AtHomePatientId"].astype(int)
poc_ids= set(bio_mapping["AtHomePatientId"])

#CONNECTED DEVICES

ww= pd.read_csv("db_withings_watch_connected.csv")
masimo= pd.read_csv("db_masimo_connected.csv")
som= pd.read_csv("db_somnoart_connected.csv")
hexa= pd.read_csv("db_hexoskin_connected.csv")
bpm= pd.read_csv("db_withings_bpm_core_connected.csv")

for df in [ww, masimo, hexa, bpm]:
    df["AtHomePatientId"] = df["AtHomePatientId"].astype(int)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    df["date"] = df["timestamp"].dt.tz_localize(None).dt.normalize()

som["AtHomePatientId"]= som["AtHomePatientId"].astype(int)
som["night_date"]= pd.to_datetime(som["night_date"], errors="coerce")

print("\nUsage:",usage.shape)
print("Intervention:",In.shape)
print("Survey all:",surv_all.shape)
print("Withings watch:",ww.shape)
print("Masimo:",masimo.shape)
print("SomnoArt:", som.shape)
print("Hexoskin:",hexa.shape)
print("BPM Core:",bpm.shape)

#CPAP BASELINES

usage["rec_n"]= usage.groupby("AtHomePatientId").cumcount()

bl= (usage[usage["rec_n"] < 56].groupby("AtHomePatientId").agg(use_mean = ("Use","mean"),
        use_std = ("Use","std"),ahi_mean = ("AHI","mean"),
        ahi_std = ("AHI","std"),leaks95_mean = ("Leaks95","mean"),n_records = ("Use","count"),).reset_index())
bl["use_std"]= bl["use_std"].fillna(0.1).clip(lower=0.1)
bl["ahi_std"]= bl["ahi_std"].fillna(0.1).clip(lower=0.1)
bl= bl.merge(usage.groupby("AtHomePatientId")["ReferenceDate"].max().reset_index(name="last_date"),on="AtHomePatientId",)
bl= bl[bl["n_records"] >= 14].reset_index(drop=True)

DEVICE_MAP= {"Resmed": 1, "Philips": 2, "Löwenstein": 3, "SEFAM": 4}

CAT_MAP= {"Unknown": 0, "Visit": 1, "Call": 2, "Sms": 3, "Not Defined": 4}

VALIDITY= {"ESS": 28, "PSQI": 28, "ISI": 28, "BDI": 42, "FSS": 42, "SF36": 84}

print(f"\nBaseline table: {len(bl):,} patients")
display(bl.head())
bl.to_csv("baselines.csv", index=False)

ESS from Monitoring3:618 rows | 618 patients
ESS from RF.csv:4,663 rows | 4,506 patients

surv_all combined:24,481 rows
ESS coverage: 5,475 patients (was 400 before fix)

Usage: (3463318, 21)
Intervention: (53513, 11)
Survey all: (24481, 4)
Withings watch: (69729, 31)
Masimo: (14428, 18)
SomnoArt: (8365, 29)
Hexoskin: (11775, 27)
BPM Core: (7479, 22)

Baseline table: 41,115 patients


,AtHomePatientId,use_mean,use_std,ahi_mean,ahi_std,leaks95_mean,n_records,last_date
0,3,5.756429,0.547349,0.176786,0.195393,26.442857,56,2025-03-09
1,15,8.108571,1.955890,2.118182,0.907192,12.109091,56,2025-03-09
2,32,6.243750,2.157224,5.476250,1.847993,NaN,56,2025-03-09
3,33,8.556607,1.661339,0.220000,0.145806,12.763636,56,2025-03-08
4,38,10.165000,1.263051,0.976786,0.467874,6.900000,56,2025-03-09


## STEP 3 — CPAP FEATURES

In [12]:
#DEVICE-SPECIFIC LEAK COMPUTATION
DEVICE_LEAK_MAP = {"Resmed":("Leaks95",24.0),"Löwenstein":("Leaks90",24.0),"SEFAM":("Leaks90",24.0),
    "Philips":("LeaksLargePercentage",30.0)}

usage["leaks_raw"]= np.nan
usage["leaks_thresh"]= np.nan

for _dev,(_col, _thresh) in DEVICE_LEAK_MAP.items():
    _mask= usage["DeviceType"] == _dev
    if _mask.sum()== 0:
        continue
    if _col in usage.columns:
        _raw= usage.loc[_mask, _col].copy()
    elif "Leaks95" in usage.columns:
        _raw= usage.loc[_mask, "Leaks95"].copy()
    else:
        _raw= pd.Series(np.nan, index=usage[_mask].index)
    if _col=="LeaksLargePercentage":
        _raw= _raw.clip(upper=100.0)
    usage.loc[_mask,"leaks_raw"]=_raw
    usage.loc[_mask,"leaks_thresh"]=_thresh

# Binary: 1 = above clinical threshold tonight
usage["leaks_high_night"]= np.where(usage["leaks_raw"].notna(),
    (usage["leaks_raw"]> usage["leaks_thresh"]).astype(float),np.nan,)

# 3-level severity: 0=low / 1=moderate / 2=high
def _leak_severity(raw, thresh):
    if pd.isna(raw) or pd.isna(thresh):
        return np.nan
    r = raw / thresh
    return 0.0 if r < 0.5 else (1.0 if r < 1.0 else 2.0)

usage["leaks_severity_night"]= usage.apply(
    lambda r: _leak_severity(r["leaks_raw"], r["leaks_thresh"]), axis=1)

# Z-score within device group
usage["leaks_zscore_night"]= np.nan
for _dev in usage["DeviceType"].unique():
    _mask= usage["DeviceType"]== _dev
    _vals= usage.loc[_mask, "leaks_raw"]
    _mu, _sd= _vals.mean(), _vals.std()
    if _sd > 0:usage.loc[_mask, "leaks_zscore_night"]=(_vals - _mu) / _sd

u_w7=usage.merge(bl[["AtHomePatientId", "last_date"]], on="AtHomePatientId", how="inner")

# Keep only records in the [0, 7]-day window before last_date.
days_ago= (u_w7["last_date"] - u_w7["ReferenceDate"]).dt.days
u_w7= u_w7[days_ago.between(0, 7)].copy()

cpap_feats= (u_w7.groupby("AtHomePatientId").agg(
        use_mean_7d= ("Use","mean"),
        use_var= ("Use","var"),
        ahi_mean_7d= ("AHI","mean"),
        ahi_var= ("AHI","var"),
        leaks_best_raw_7d= ("leaks_raw","mean"),
        leaks_high_7d= ("leaks_high_night","mean"),
        leaks_severity_7d= ("leaks_severity_night","mean"),
        leaks_zscore_dev= ("leaks_zscore_night","mean"),
        leaks_worst_7d= ("leaks_raw","max"),
        leaks_var_7d= ("leaks_raw","var"),
        nights_valid= ("Use",lambda x: (x > 0).sum()),
        pressure_mean= ("Presure90","mean"),).reset_index())

# backward-compat alias for DROPOUT_FEATS / COX_FEATS
cpap_feats["leaks95_mean_7d"]= cpap_feats["leaks_high_7d"]
cpap_feats["leaks95_var"]= cpap_feats["leaks_var_7d"]

#USE SLOPE

cpap_feats= cpap_feats.merge(
    u_w7.groupby("AtHomePatientId")["Use"].apply(lambda s: float(np.polyfit(range(len(s)), s.values, 1)[0]) if len(s) >= 3 else 0.0)
    .reset_index(name="use_slope"),on="AtHomePatientId",how="left",)

#LOW-USE STREAK

def low_streak(s):
    """Longest consecutive run of nights with Use < 4 h."""
    mx= cur = 0
    for v in s.values:
        cur= cur + 1 if v < 4.0 else 0
        mx= max(mx, cur)
    return mx

cpap_feats= cpap_feats.merge(
    u_w7.groupby("AtHomePatientId")["Use"]
    .apply(low_streak)
    .reset_index(name="low_use_streak"),
    on="AtHomePatientId",
    how="left",)

#INSTABILITY INDEX

cpap_feats= cpap_feats.merge(
    u_w7.groupby("AtHomePatientId")["Use"]
    .apply(lambda s: float(np.mean(np.abs(s.values - s.mean()) > 1.0)))
    .reset_index(name="instability_idx"),
    on="AtHomePatientId",
    how="left",)

#Z-SCORES

bl_ref= bl[["AtHomePatientId", "use_mean", "use_std", "ahi_mean", "ahi_std"]].copy()
bl_ref.columns= ["AtHomePatientId", "bl_use_mean", "bl_use_std", "bl_ahi_mean", "bl_ahi_std"]

cpap_feats= cpap_feats.merge(bl_ref, on="AtHomePatientId", how="left")

# use_mean_7d / ahi_mean_7d
cpap_feats["use_zscore"]= (
    (cpap_feats["use_mean_7d"]- cpap_feats["bl_use_mean"])
    / (cpap_feats["bl_use_std"] + 1e-6))
cpap_feats["ahi_zscore"]= (
    (cpap_feats["ahi_mean_7d"]- cpap_feats["bl_ahi_mean"])
    / (cpap_feats["bl_ahi_std"] + 1e-6))

cpap_feats["pressure_missing"]= cpap_feats["pressure_mean"].isna().astype(int)
cpap_feats= cpap_feats.drop(columns=["bl_use_mean", "bl_use_std", "bl_ahi_mean", "bl_ahi_std"])


_u_sorted= usage[usage["AtHomePatientId"].isin(bl["AtHomePatientId"])][["AtHomePatientId", "ReferenceDate"]
].sort_values(["AtHomePatientId", "ReferenceDate"])
_recent_gap= (_u_sorted.groupby("AtHomePatientId")["ReferenceDate"].apply(
    lambda s: int((s.iloc[-1] - s.iloc[-2]).days) if len(s) >= 2 else 0)
    .reset_index(name="recent_gap"))
_recent_gap["recent_gap"] = pd.to_numeric(_recent_gap["recent_gap"], errors="coerce").fillna(0).astype(int)
cpap_feats= cpap_feats.merge(_recent_gap, on="AtHomePatientId",how="left")
cpap_feats["transmission_gap"] = cpap_feats["recent_gap"].fillna(0).astype(int).clip(lower=0)
cpap_feats= cpap_feats.drop(columns=["recent_gap"])

print(f"CPAP block: {len(cpap_feats):,} patients × {len(cpap_feats.columns)} features")
print(f"Key renamed columns present: "
      f"use_mean_7d={('use_mean_7d' in cpap_feats.columns)},"
      f"ahi_mean_7d={('ahi_mean_7d' in cpap_feats.columns)},"
      f"leaks95_mean_7d={('leaks95_mean_7d' in cpap_feats.columns)}")
display(cpap_feats.head())
cpap_feats.to_csv("cpap_features.csv", index=False)

CPAP block: 41,115 patients × 22 features
Key renamed columns present: use_mean_7d=True,ahi_mean_7d=True,leaks95_mean_7d=True


,AtHomePatientId,use_mean_7d,use_var,ahi_mean_7d,ahi_var,leaks_best_raw_7d,leaks_high_7d,leaks_severity_7d,leaks_zscore_dev,leaks_worst_7d,...,pressure_mean,leaks95_mean_7d,leaks95_var,use_slope,low_use_streak,instability_idx,use_zscore,ahi_zscore,pressure_missing,transmission_gap
0,3,5.19750,0.222336,0.0125,0.001250,41.70,1.000,2.000,1.312698,55.2,...,NaN,1.000,54.205714,-0.000476,0,0.000,-1.021154,-0.840790,1,1
1,15,7.47500,4.641057,1.5125,0.818393,18.60,0.250,0.875,0.032764,40.8,...,NaN,0.250,188.845714,-0.486429,1,0.375,-0.323930,-0.667644,1,1
2,32,6.07375,4.379255,5.3550,10.919286,15.50,0.125,0.375,0.295554,62.0,...,10.00,0.125,411.714286,0.075357,1,1.000,-0.078805,-0.065612,0,1
3,33,9.28750,0.128336,0.2125,0.006964,16.05,0.250,0.875,-0.108528,30.0,...,11.76,0.250,90.077143,-0.045476,0,0.000,0.439942,-0.051438,0,1
4,38,10.44875,0.087127,0.9000,0.065714,11.40,0.000,0.625,-0.366177,15.6,...,NaN,0.000,16.868571,-0.074167,0,0.000,0.224654,-0.164116,1,1


## STEP 4 — SURVEY FEATURES

In [14]:
surv_feats= bl[["AtHomePatientId", "last_date"]].copy()

for survey,window in VALIDITY.items():

    rows= surv_all[surv_all["survey_name"] == survey].copy()

    rows= rows.merge(bl[["AtHomePatientId", "last_date"]], on="AtHomePatientId", how="inner")

    last= (rows.sort_values("execution_date")
        .drop_duplicates(subset="AtHomePatientId", keep="last").reset_index(drop=True))

    # Freshness: 1.0 = collected today, 0.0 = at or beyond validity window
    last["days_old"]= (last["last_date"] - last["execution_date"]).dt.days.clip(lower=0)
    last["freshness"]= (1.0 - last["days_old"] / window).clip(lower=0.0, upper=1.0)

    last= last.rename(columns={"score_value": f"{survey}_score","freshness":   f"{survey}_freshness",})
    last[f"{survey}_missing"]= 0

    surv_feats= surv_feats.merge(last[["AtHomePatientId", f"{survey}_score",
              f"{survey}_freshness", f"{survey}_missing"]],on="AtHomePatientId",how="left",)

    # Patients absent from this survey: missing=1, freshness=0, score stays NaN.
    surv_feats[f"{survey}_missing"]= surv_feats[f"{survey}_missing"].fillna(1).astype(int)
    surv_feats[f"{survey}_freshness"]= surv_feats[f"{survey}_freshness"].fillna(0.0)

surv_feats = surv_feats.drop(columns=["last_date"])

print(f"Survey block: {len(surv_feats):,} patients × {len(surv_feats.columns)} features")
print("\nCoverage per scale (patients with score):")
for survey in VALIDITY:
    n= surv_feats[f"{survey}_missing"].eq(0).sum()
    pct= n / len(surv_feats) * 100
    print(f"{survey:<6}: {n:>6,}({pct:.1f}%)")

display(surv_feats.head())
surv_feats.to_csv("survey_features.csv", index=False)

Survey block: 41,115 patients × 19 features

Coverage per scale (patients with score):
ESS   :  4,837(11.8%)
PSQI  :    400(1.0%)
ISI   :    400(1.0%)
BDI   :    400(1.0%)
FSS   :    400(1.0%)
SF36  :    400(1.0%)


,AtHomePatientId,ESS_score,ESS_freshness,ESS_missing,PSQI_score,PSQI_freshness,PSQI_missing,ISI_score,ISI_freshness,ISI_missing,BDI_score,BDI_freshness,BDI_missing,FSS_score,FSS_freshness,FSS_missing,SF36_score,SF36_freshness,SF36_missing
0,3,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1
1,15,6.0,0.0,0,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1
2,32,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1
3,33,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1
4,38,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1,NaN,0.0,1


## STEP 5 — CARE FEATURES

In [17]:
#CARE FEATURES — interventions in last 30/90 days
In_w= In.merge(bl[["AtHomePatientId", "last_date"]],on = "AtHomePatientId",
    how = "inner",)
In_w["days_ago"]= (In_w["last_date"] - In_w["ReferenceDate"]).dt.days

In_90= In_w[In_w["days_ago"].between(0, 90)]
In_30= In_w[In_w["days_ago"].between(0, 30)]

done_90= In_90[In_90["Status"]== "Done"]
pend_90= In_90[In_90["Status"]== "Pending"]
done_30= In_30[In_30["Status"] == "Done"]

#AGGREGATE FEATURES

care_feats= (bl[["AtHomePatientId"]].merge(
        done_90.groupby("AtHomePatientId").agg(intervention_count_90d = ("ReferenceDate", "count"),
            intervention_diversity= ("Category","nunique"),days_since_last = ("days_ago","min"),).reset_index(),
        on="AtHomePatientId",how="left",).merge(done_30.groupby("AtHomePatientId").agg(
            intervention_count_30d= ("ReferenceDate", "count"),
            visit_count_30d= ("Channel", lambda x: (x == "Visit").sum()),
            call_count_30d= ("Channel", lambda x: (x == "Call").sum()),
            sms_count_30d= ("Channel", lambda x: (x == "SMS").sum()),
        ).reset_index(),on="AtHomePatientId", how="left",).merge(
        pend_90.groupby("AtHomePatientId").agg(pending_count = ("ReferenceDate", "count"),
            pending_max_age = ("days_ago","max"),).reset_index(),on="AtHomePatientId", how="left",))

#LAST 3 INTERVENTION SEQUENCE

def last3_enc(grp):
    """Encode the 3 most recent intervention categories as integer sequence."""
    vals= grp.sort_values("ReferenceDate").tail(3)["Category"].values
    t= [CAT_MAP.get(str(v), 0) for v in vals]
    while len(t) < 3:
        t.insert(0, 0)
    return pd.Series({
        "last_intervention_t1": t[-3],"last_intervention_t2": t[-2],"last_intervention_t3": t[-1],})

care_feats = care_feats.merge(done_90.groupby("AtHomePatientId")[["ReferenceDate", "Category"]]
           .apply(last3_enc).reset_index(),on="AtHomePatientId",how="left",)

care_feats = care_feats.fillna({
    "intervention_count_90d":0,
    "intervention_diversity":0,
    "days_since_last": 999,
    "intervention_count_30d":0,
    "visit_count_30d":0,
    "call_count_30d":0,
    "sms_count_30d":0,
    "pending_count":0,
    "pending_max_age":0,
    "last_intervention_t1":0,
    "last_intervention_t2":0,
    "last_intervention_t3":0,})

#RESPONSE RATE

_triplet_file="layer4_triplets.csv"
try:
    _t= pd.read_csv(_triplet_file, usecols=["AtHomePatientId", "delta_use"])
    _resp= (_t.groupby("AtHomePatientId")["delta_use"]
        .agg(lambda x: float((x > 0.5).mean()))
        .reset_index(name="response_rate"))
    care_feats= care_feats.merge(_resp, on="AtHomePatientId", how="left")
    care_feats["response_rate"]= care_feats["response_rate"].fillna(0.5)
    print(f"response_rate: from {_triplet_file} "
          f"({_resp['AtHomePatientId'].nunique():,} patients, "
          f"mean={_resp['response_rate'].mean():.3f})")
except FileNotFoundError:care_feats["response_rate"] = 0.5
print("response_rate: defaulted to 0.5 ""(run Layer 4 first, then re-run this cell for real values)")

#CARE MISSING FLAG

care_feats["care_missing"]= (care_feats["intervention_count_90d"] == 0).astype(int)

print(f"\nCare block: {len(care_feats):,} patients × {len(care_feats.columns)} features")
print(f"care_missing=1 (no contact in 90d): {care_feats['care_missing'].sum():,} "
      f"({care_feats['care_missing'].mean()*100:.1f}%)")
display(care_feats.head())
care_feats.to_csv("care_features.csv", index=False)

response_rate: from layer4_triplets.csv (16,038 patients, mean=0.255)
response_rate: defaulted to 0.5 (run Layer 4 first, then re-run this cell for real values)

Care block: 41,115 patients × 15 features
care_missing=1 (no contact in 90d): 22,912 (55.7%)


,AtHomePatientId,intervention_count_90d,intervention_diversity,days_since_last,intervention_count_30d,visit_count_30d,call_count_30d,sms_count_30d,pending_count,pending_max_age,last_intervention_t1,last_intervention_t2,last_intervention_t3,response_rate,care_missing
0,3,2.0,1.0,27.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0
1,15,0.0,0.0,999.0,0.0,0.0,0.0,0.0,1.0,58.0,0.0,0.0,0.0,0.5,1
2,32,0.0,0.0,999.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.5,1
3,33,0.0,0.0,999.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,1
4,38,0.0,0.0,999.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,1


## STEP 6 — BIO FEATURES

In [18]:
#BIO FEATURES — PoC patients only

poc_ref_base= (bl[bl["AtHomePatientId"].isin(poc_ids)][["AtHomePatientId", "last_date"]].copy().reset_index(drop=True))

#BIO WINDOW FUNCTION

def bio_window(df, col="date"):
    """
    Return rows from `df` within 7 days of each patient's own device
    last-observation date.  Falls back to CPAP last_date if no device data.
    """
    # Normalise to tz-naive
    col_ts= pd.to_datetime(df[col])
    if col_ts.dt.tz is not None:
        col_ts= col_ts.dt.tz_convert(None)

    # Per-device, per-patient last observation date
    dev_last = (df[["AtHomePatientId"]].assign(_ts=col_ts).groupby("AtHomePatientId")["_ts"].max().reset_index(name="dev_last"))

    d= df.copy()
    d["_ts"] = col_ts
    d= d.merge(poc_ref_base, on="AtHomePatientId", how="inner")
    d= d.merge(dev_last,on="AtHomePatientId", how="left")
    d["anchor"]= d["dev_last"].fillna(d["last_date"])
    d["days_ago"]= (d["anchor"] - d["_ts"]).dt.days
    return d[d["days_ago"].between(0, 7)].drop(columns=["dev_last", "anchor", "_ts"])

#APPLY WINDOW TO EACH DEVICE

ww_7= bio_window(ww)
mas_7= bio_window(masimo)
hex_7= bio_window(hexa)
som_7= bio_window(som, col="night_date")
bpm_7= bio_window(bpm)

som_v = (som_7[som_7["analysis_status"] == "Valid"]
    if "analysis_status" in som_7.columns
    else pd.DataFrame())

#AGGREGATE BIO FEATURES

bio_feats= poc_ref_base[["AtHomePatientId"]].copy()

bio_feats= bio_feats.merge(
    ww_7.groupby("AtHomePatientId").agg(
        hrv_rmssd_7d = ("hrv_rmssd","mean"),
        spo2_ww_7d = ("spo2","mean"),
        sleep_efficiency_7d=("sleep_efficiency","mean"),
        sleep_score_7d = ("sleep_score","mean"),
        snoring_s_7d = ("snoring_s","mean"),
        wakeup_count_7d = ("wakeup_count","mean"),).reset_index(),
    on="AtHomePatientId", how="left",)
bio_feats = bio_feats.merge(mas_7.groupby("AtHomePatientId").agg(
        spo2_masimo_7d = ("spo2","mean"),
        pvi_7d = ("pleth_variability_index",  "mean"),).reset_index(),on="AtHomePatientId", how="left",)
bio_feats = bio_feats.merge(som_v.groupby("AtHomePatientId").agg(
        tst_min_7d = ("tst_min","mean"),
        waso_min_7d = ("waso_min","mean"),
        n3_duration_7d = ("n3_duration_min","mean"),
        rem_duration_7d = ("rem_duration_min","mean"),
        nb_awakenings_7d= ("nb_awakenings","mean"),).reset_index(),on="AtHomePatientId", how="left",)
bio_feats = bio_feats.merge(
    hex_7.groupby("AtHomePatientId").agg(
        hrv_lf_7d = ("hrv_lf","mean"),
        hrv_hf_7d = ("hrv_hf","mean"),
        breathing_rate_7d=("breathing_rate","mean"),).reset_index(),on="AtHomePatientId", how="left",)
bio_feats = bio_feats.merge(
    bpm_7.groupby("AtHomePatientId").agg(
        systolic_bp_7d= ("systolic_bp","mean"),
        diastolic_bp_7d= ("diastolic_bp","mean"),
        afib_detected= ("afib_result","max"),).reset_index(),on="AtHomePatientId", how="left",)

#DERIVED FEATURES

bio_feats["hrv_lf_hf_ratio_7d"]= bio_feats["hrv_lf_7d"] / (bio_feats["hrv_hf_7d"] + 1e-6)

spo2_cols= [c for c in ["spo2_ww_7d", "spo2_masimo_7d"] if c in bio_feats.columns]
bio_feats["spo2_7d"]= bio_feats[spo2_cols].mean(axis=1, skipna=True)

#DEVICE PRESENCE FLAGS

bio_feats["ww_absent"]= bio_feats["hrv_rmssd_7d"].isna().astype(int)
bio_feats["masimo_absent"]= bio_feats["spo2_masimo_7d"].isna().astype(int)
bio_feats["somnoart_absent"]= bio_feats["tst_min_7d"].isna().astype(int)
bio_feats["hexoskin_absent"]= bio_feats["hrv_lf_hf_ratio_7d"].isna().astype(int)
bio_feats["bpm_absent"]= bio_feats["systolic_bp_7d"].isna().astype(int)
bio_feats["n_devices"]= (5 - bio_feats[["ww_absent","masimo_absent","somnoart_absent","hexoskin_absent","bpm_absent"]].sum(axis=1))
bio_feats["bio_absent"]= (bio_feats["n_devices"] == 0).astype(int)
bio_feats["afib_detected"]= bio_feats["afib_detected"].fillna(0).astype(int)

bio_feats= bio_feats.drop(columns=["hrv_lf_7d", "hrv_hf_7d"], errors="ignore")

print(f"Bio block: {len(bio_feats):,} PoC patients × {len(bio_feats.columns)} features")
print("\nDevice coverage:")
for flag, name in [("ww_absent","Withings Watch"),("masimo_absent","Masimo"),
                   ("somnoart_absent","SomnoArt"),("hexoskin_absent","Hexoskin"),
                   ("bpm_absent","BPM Core")]:
    n = (bio_feats[flag] == 0).sum()
    print(f"{name:<20}: {n:>4} patients with data ({n/len(bio_feats)*100:.1f}%)")

display(bio_feats.head())
bio_feats.to_csv("bio_features.csv", index=False)
print(f"Exported: bio_features.csv|{bio_feats.shape[0]:,}rows×{bio_feats.shape[1]} cols")

Bio block: 400 PoC patients × 27 features

Device coverage:
Withings Watch      :  357 patients with data (89.2%)
Masimo              :  271 patients with data (67.8%)
SomnoArt            :  182 patients with data (45.5%)
Hexoskin            :  150 patients with data (37.5%)
BPM Core            :  187 patients with data (46.8%)


,AtHomePatientId,hrv_rmssd_7d,spo2_ww_7d,sleep_efficiency_7d,sleep_score_7d,snoring_s_7d,wakeup_count_7d,spo2_masimo_7d,pvi_7d,tst_min_7d,...,afib_detected,hrv_lf_hf_ratio_7d,spo2_7d,ww_absent,masimo_absent,somnoart_absent,hexoskin_absent,bpm_absent,n_devices,bio_absent
0,256,35.512250,92.055000,0.629250,62.125,610.800000,4.250000,92.893333,13.423333,NaN,...,0,1.082647,92.474167,0,0,1,0,0,4,0
1,2297,26.511400,92.366000,0.685600,59.000,443.460000,2.800000,NaN,NaN,NaN,...,0,1.340727,92.366000,0,1,1,0,1,2,0
2,2737,17.440333,87.716667,0.555000,39.000,1301.066667,6.666667,87.835000,20.685000,NaN,...,0,NaN,87.775833,0,0,1,1,0,3,0
3,2872,12.037167,88.125000,0.482667,49.500,1123.233333,8.333333,NaN,NaN,NaN,...,0,NaN,88.125000,0,1,1,1,0,2,0
4,3347,44.577429,90.342857,0.772571,76.000,226.371429,1.000000,NaN,NaN,384.616,...,0,0.872146,90.342857,0,1,0,0,0,4,0


Exported: bio_features.csv|400rows×27 cols


## STEP 7 — FEATURES MERGE

In [22]:
features= bl[["AtHomePatientId","last_date","use_mean","use_std","ahi_mean","ahi_std",]].copy()

features= features.merge(cpap_feats,on="AtHomePatientId", how="left")
features= features.merge(surv_feats,on="AtHomePatientId", how="left")
features= features.merge(care_feats,on="AtHomePatientId", how="left")
features= features.merge(bio_feats,on="AtHomePatientId", how="left")

#COLLISION GUARD

_collisions= [c for c in features.columns if c.endswith("_x") or c.endswith("_y")]
assert len(_collisions)== 0,(f"\nColumn collision detected after merge: {_collisions}\n"
    "This means cpap_feats still has the old names (use_mean / ahi_mean / leaks95_mean).\n"
    "Re-run Step 3 (cpap_features block) with the renamed 7d columns:\n"
    "  use_mean → use_mean_7d,  ahi_mean → ahi_mean_7d,  leaks95_mean → leaks95_mean_7d")

#FLAGS

features["in_poc"]= features["AtHomePatientId"].isin(poc_ids)
features["bio_absent"]= features["bio_absent"].fillna(1).astype(int)
features["n_devices"]= features["n_devices"].fillna(0).astype(int)

_cpap_col= "use_mean_7d" if "use_mean_7d" in features.columns else "use_mean"

print(f"Feature matrix:{len(features):,} patients × {features.shape[1]} columns")
print(f"\nBlock coverage:")
print(f" PAP features ({_cpap_col}): "
      f"{features[_cpap_col].notna().sum():,} patients")
print(f"ESS survey present:          "
      f"{(features['ESS_missing']==0).sum():,} patients")
print(f"Any intervention in 90d:     "
      f"{(features['care_missing']==0).sum():,} patients")
print(f"PoC / bio data present:"f"{features['in_poc'].sum():,} / {(features['bio_absent']==0).sum():,} patients")

display(features.head())
features.to_csv("features_merged.csv", index=False)

Feature matrix:41,115 patients × 86 columns

Block coverage:
 PAP features (use_mean_7d): 41,115 patients
ESS survey present:          4,837 patients
Any intervention in 90d:     18,203 patients
PoC / bio data present:400 / 396 patients


,AtHomePatientId,last_date,use_mean,use_std,ahi_mean,ahi_std,use_mean_7d,use_var,ahi_mean_7d,ahi_var,...,hrv_lf_hf_ratio_7d,spo2_7d,ww_absent,masimo_absent,somnoart_absent,hexoskin_absent,bpm_absent,n_devices,bio_absent,in_poc
0,3,2025-03-09,5.756429,0.547349,0.176786,0.195393,5.19750,0.222336,0.0125,0.001250,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,False
1,15,2025-03-09,8.108571,1.955890,2.118182,0.907192,7.47500,4.641057,1.5125,0.818393,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,False
2,32,2025-03-09,6.243750,2.157224,5.476250,1.847993,6.07375,4.379255,5.3550,10.919286,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,False
3,33,2025-03-08,8.556607,1.661339,0.220000,0.145806,9.28750,0.128336,0.2125,0.006964,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,False
4,38,2025-03-09,10.165000,1.263051,0.976786,0.467874,10.44875,0.087127,0.9000,0.065714,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,False


## STEP 8 — EVIDENCE STATE (presence, freshness, reliability, routing)

In [23]:
_cpap_col = (
    "use_mean_7d" if "use_mean_7d" in features.columns
    else "use_mean_x" if "use_mean_x" in features.columns
    else "use_mean")
features["pi_cpap"] = (features[_cpap_col].notna()& (features["nights_valid"].fillna(0) > 0)).astype(int)

# All 6 survey scales—dynamically resolved so missing scales don't crash.
_miss_cols= [f"{s}_missing" for s in ("ESS","PSQI","ISI","BDI","FSS","SF36")
              if f"{s}_missing" in features.columns]
features["pi_surv"]= (features[_miss_cols] == 0).any(axis=1).astype(int)

features["pi_care"]= (features["care_missing"] == 0).astype(int)
features["pi_bio"]= (features["bio_absent"]   == 0).astype(int)

#FRESHNESS/RELIABILITY (phi)

features["phi_cpap"]= (1.0 - features["transmission_gap"].fillna(7) / 7.0).clip(0.0, 1.0)

_fresh_cols= [f"{s}_freshness" for s in ("ESS","PSQI","ISI","BDI","FSS","SF36")
               if f"{s}_freshness" in features.columns]
features["phi_surv"] = features[_fresh_cols].mean(axis=1, skipna=True).fillna(0.0)

features["phi_care"]= (1.0 - features["days_since_last"].fillna(999) / 90.0).clip(0.0, 1.0)

features["phi_bio"]= (features["n_devices"] / 5.0).clip(0.0, 1.0)

#COMPOSITE RELIABILITY (rho)
# Weights: CPAP 0.45 + Survey 0.20 + Care 0.25 + Bio 0.10 = 1.00
features["rho"]= (0.45 * features["pi_cpap"] * features["phi_cpap"].clip(0, 1)
    + 0.20 * features["pi_surv"] * features["phi_surv"].clip(0, 1)+ 0.25 * features["pi_care"] * features["phi_care"]
    + 0.10 * features["pi_bio"]  * features["phi_bio"].clip(0, 1)).round(4)

features["routing"]= np.select(
    condlist=[
        features["pi_cpap"] == 0,      # no recent CPAP data → technician
        features["pi_care"] == 0,       # never contacted in 90d → sync
        features["pi_surv"] == 0,       # no survey on file → reminder
        features["phi_cpap"] < 0.5,     # stale transmission → warning
    ],
    choicelist=[
        "technician_alert",
        "sync_alert",
        "survey_reminder",
        "proceed_with_warning",],default="proceed",)

print(f"Feature matrix with evidence: {len(features):,} patients × {features.shape[1]} columns")
print(f"(pi_cpap column source: '{_cpap_col}')")

print("\nRouting distribution:")
print(features["routing"].value_counts().to_string())

print(f"\nMean reliability (rho): {features['rho'].mean():.3f}")
print(f"rho < 0.3:{(features['rho'] < 0.3).sum():,} patients")
print(f"rho >= 0.7:{(features['rho'] >= 0.7).sum():,} patients")

print("\nFreshness means:")
for col in ["phi_cpap","phi_surv","phi_care","phi_bio"]:
    print(f"{col}:{features[col].mean():.3f}")

features.to_csv("features_with_evidence.csv", index=False)
print("\nSaved: features_with_evidence.csv")

Feature matrix with evidence: 41,115 patients × 96 columns
(pi_cpap column source: 'use_mean_7d')

Routing distribution:
routing
sync_alert              22465
survey_reminder         14642
proceed                  3250
technician_alert          743
proceed_with_warning       15

Mean reliability (rho): 0.449
rho < 0.3:1,014 patients
rho >= 0.7:164 patients

Freshness means:
phi_cpap:0.850
phi_surv:0.010
phi_care:0.284
phi_bio:0.006

Saved: features_with_evidence.csv


## STEP 9 — DISPLAY & SAVE

In [25]:
#SAMPLE DISPLAY — 5 PoC patients

print("\nSample — 5 PoC patients:")
poc_s= features[features["in_poc"]].head(5)

if "use_mean_7d" in poc_s.columns:
    use_col = "use_mean_7d"
elif "use_mean_x" in poc_s.columns:
    use_col= "use_mean_x"
else:
    use_col= "use_mean"

sample_cols= [
    "AtHomePatientId",
    use_col,
    "use_zscore",
    "ESS_score",
    "ESS_freshness",
    "intervention_count_30d",
    "days_since_last",
    "spo2_7d",
    "hrv_rmssd_7d",
    "rho",
    "routing",]
sample_cols= [c for c in sample_cols if c in poc_s.columns]
print(poc_s[sample_cols].to_string(index=False))

features.to_csv("features_with_evidence.csv", index=False)
print(f"\nSaved: features_with_evidence.csv|"f"{features.shape[0]:,}rows × {features.shape[1]} cols")


Sample — 5 PoC patients:
 AtHomePatientId  use_mean_7d  use_zscore  ESS_score  ESS_freshness  intervention_count_30d  days_since_last   spo2_7d  hrv_rmssd_7d    rho    routing
             256       5.4225   -1.246444       11.0            1.0                     0.0            999.0 92.474167     35.512250 0.6657 sync_alert
            2297      10.0025   -0.111067       10.0            1.0                     0.0            999.0 92.366000     26.511400 0.6257 sync_alert
            2737       7.8650    0.207220       23.0            1.0                     0.0            999.0 87.775833     17.440333 0.6457 sync_alert
            2872       3.8200   -0.035217       18.0            1.0                     5.0              2.0 88.125000     12.037167 0.8702    proceed
            3347       7.5000    0.809921        3.0            1.0                     0.0             44.0 90.342857     44.577429 0.7935    proceed

Saved: features_with_evidence.csv|41,115rows × 96 cols


## LAYER 2 — SETUP (imports, feature lists)

In [26]:
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
    XGB_OK= True
except ImportError:
    XGB_OK= False

try:
    from lightgbm import LGBMClassifier
    LGB_OK= True
except ImportError:
    LGB_OK= False

try:
    from catboost import CatBoostClassifier
    CAT_OK= True
except ImportError:
    CAT_OK= False

SEED= 42

#CPAP FEATURES
CPAP_FEATS= [
    "use_mean_7d",
    "use_var",
    "use_slope",
    "ahi_mean_7d",
    "ahi_var",
    "leaks95_mean_7d",    
    "leaks_high_7d",      
    "leaks_severity_7d",  
    "leaks_zscore_dev",   
    "leaks_worst_7d",     
    "leaks_var_7d",       
    "low_use_streak",
    "instability_idx",
    "use_zscore",
    "ahi_zscore",
    "transmission_gap",
    "nights_valid",
    "pressure_mean",     
    "pressure_missing",]

#SURVEY FEATURES

SURVEY_FEATS= [
    "ESS_score",  "ESS_freshness",  "ESS_missing",
    "PSQI_score", "PSQI_freshness", "PSQI_missing",
    "BDI_score",  "BDI_freshness",  "BDI_missing",
    "ISI_score",  "ISI_freshness",  "ISI_missing",
    "FSS_score",  "FSS_freshness",  "FSS_missing",
    "SF36_score", "SF36_freshness", "SF36_missing",]

#CARE FEATURES

CARE_FEATS= [
    "intervention_count_30d",
    "intervention_count_90d",
    "intervention_diversity",
    "pending_count",
    "pending_max_age",
    "days_since_last",
    "visit_count_30d",
    "call_count_30d",
    "sms_count_30d",
    "last_intervention_t1",
    "last_intervention_t2",
    "last_intervention_t3",
    "response_rate",
    "care_missing",]

#BIO FEATURES

BIO_FEATS= [
    "hrv_rmssd_7d",
    "spo2_7d",
    "sleep_efficiency_7d",
    "sleep_score_7d",
    "snoring_s_7d",
    "wakeup_count_7d",
    "spo2_masimo_7d",
    "pvi_7d",
    "tst_min_7d",
    "waso_min_7d",
    "n3_duration_7d",
    "rem_duration_7d",
    "nb_awakenings_7d",
    "hrv_lf_hf_ratio_7d",
    "breathing_rate_7d",
    "systolic_bp_7d",
    "diastolic_bp_7d",
    "afib_detected",
    "ww_absent",
    "masimo_absent",
    "somnoart_absent",
    "hexoskin_absent",
    "bpm_absent",
    "n_devices",]

print("Layer 2 setup complete.")
print(f"CPAP features:{len(CPAP_FEATS)}")
print(f"Survey features:{len(SURVEY_FEATS)}")
print(f"Care features:{len(CARE_FEATS)}")
print(f"Bio features:{len(BIO_FEATS)}")
print(f"\nXGBoost:{XGB_OK}")
print(f"LightGBM:{LGB_OK}")
print(f"CatBoost:{CAT_OK}")

Layer 2 setup complete.
CPAP features:19
Survey features:18
Care features:14
Bio features:24

XGBoost:True
LightGBM:False
CatBoost:False


## STEP 11 — FEATURE AVAILABILITY CHECK

In [29]:
#FEATURE AVAILABILITY CHECK

if "features" not in globals():
    raise RuntimeError(
        "The dataframe `features` is not available. "
        "Run the Layer 1 harmonization / feature-matrix code first.")

print(f"Input feature matrix:{len(features):,} patients × {features.shape[1]} columns")

available_cpap=[c for c in CPAP_FEATS if c in features.columns]
available_survey=[c for c in SURVEY_FEATS if c in features.columns]
available_care=[c for c in CARE_FEATS if c in features.columns]
available_bio=[c for c in BIO_FEATS if c in features.columns]

print(f"\nAvailable CPAP features:{len(available_cpap)}/{len(CPAP_FEATS)}")
print(f"Available survey features:{len(available_survey)}/{len(SURVEY_FEATS)}")
print(f"Available care features:{len(available_care)}/{len(CARE_FEATS)}")
print(f"Available biomarker features:{len(available_bio)}/{len(BIO_FEATS)}")

#MISSING FEATURE WARNINGS

_b1_cols= {"use_mean_7d", "ahi_mean_7d", "leaks95_mean_7d", "leaks_high_7d"}
_warnings= []

for feat_name, available, full_list in [
    ("CPAP",available_cpap,CPAP_FEATS),
    ("Survey",available_survey,SURVEY_FEATS),
    ("Care",available_care,CARE_FEATS),
    ("Bio", available_bio,BIO_FEATS),
]:
    missing= [c for c in full_list if c not in features.columns]
    if missing:
        _warnings.append((feat_name, missing))

if _warnings:
    print("\n MISSING FEATURES — re-run the indicated upstream steps:")
    for feat_name, missing in _warnings:
        b1_hit=[c for c in missing if c in _b1_cols]
        hint= " ← re-run Step 3 (cpap_feats B1 rename fix)" if b1_hit else ""
        print(f"[{feat_name}] {missing}{hint}")
else:
    print("\n All feature lists fully resolved in the feature matrix.")

#NaN COVERAGE PER BLOCK

print("\nPatient coverage per block (at least 1 non-NaN feature):")
for label, cols in [("CPAP",available_cpap),
                    ("Survey",available_survey),
                    ("Care",available_care),
                    ("Bio",available_bio)]:
    if not cols:
        print(f"{label:<8}: 0 features available — skipping")
        continue
    n_covered= features[cols].notna().any(axis=1).sum()
    pct= n_covered / len(features) * 100
    print(f"{label:<8}: {n_covered:>6,} / {len(features):,}  ({pct:.1f}%)")

Input feature matrix:41,115 patients × 96 columns

Available CPAP features:19/19
Available survey features:18/18
Available care features:14/14
Available biomarker features:24/24

 All feature lists fully resolved in the feature matrix.

Patient coverage per block (at least 1 non-NaN feature):
CPAP    : 41,115 / 41,115  (100.0%)
Survey  : 41,115 / 41,115  (100.0%)
Care    : 41,115 / 41,115  (100.0%)
Bio     : 41,115 / 41,115  (100.0%)


## LAYER 2 — SPECIALIST CLASSES

In [30]:
class CPAPSpecialist:
    MACRO_STATES={
        0:"stable_adherent",
        1:"moderate_user",
        2:"declining_user",
        3:"critical_non_adherent",}

    def __init__(self):
        self.phase="macro_rules"
        self.cluster_history= []
        self.stability_window= 3
        self.ms_models= {}
        self.classifier= None
        self.classifier_name= ""
        self.scaler= StandardScaler()
        self.K_sub= {}

    def assign_macro_state(self, df):
        use_col=("use_mean_7d" if "use_mean_7d" in df.columns
                   else "use_mean_x" if "use_mean_x" in df.columns
                   else "use_mean")
        ahi_col=("ahi_mean_7d" if "ahi_mean_7d" in df.columns
                   else "ahi_mean_x" if "ahi_mean_x" in df.columns
                   else "ahi_mean")
        use=df[use_col].fillna(0)
        ahi=df[ahi_col].fillna(10)
        slp=df["use_slope"].fillna(0)
        low=df["low_use_streak"].fillna(0)

        labels=np.ones(len(df), dtype=int)
        labels[
            (use >= 6.0) &
            (ahi <= 10)  &
            (slp > -0.1) &
            (low== 0)] = 0
        labels[
            (slp < -0.3) |
            (low >= 2)
        ]= 2

        labels[
            (use < 2.0) |
            (low >= 3)
        ]= 3

        return labels

    def fit_clustering(self, df, macro_labels, target_states=None):
        if target_states is None:
            target_states= [1, 2, 3]

        cpap_cols= [c for c in CPAP_FEATS if c in df.columns]

        X_all= df[cpap_cols].fillna(0).values
        self.scaler.fit(X_all)

        all_labels= macro_labels.copy()
        results= {}

        for state in target_states:
            mask= macro_labels== state

            if mask.sum() < 50:
                continue

            X_sub= df[mask][cpap_cols].fillna(0).values
            X_s= self.scaler.transform(X_sub)

            bw= estimate_bandwidth(
                X_s,
                quantile= 0.5,
                n_samples= min(3000, len(X_s)),
                random_state= SEED,
                n_jobs= -1,)
            
            ms= MeanShift(bandwidth=bw, bin_seeding=True, n_jobs=-1)
            ms.fit(X_s)

            K_sub= len(np.unique(ms.labels_))

            sil= (
                silhouette_score(
                    X_s, ms.labels_,
                    sample_size = min(3000, len(X_s)),
                    random_state = SEED,)
                if K_sub > 1 else 0.0
            )

            offset= state * 100
            all_labels[mask]= ms.labels_ + offset

            self.ms_models[state]= ms
            self.K_sub[state]= K_sub

            results[state]= {"K": K_sub, "silhouette": sil, "n": mask.sum()}

            print(f"  State {state} ({self.MACRO_STATES[state]}): "
                f"K_sub={K_sub}  sil={sil:.3f}  n={mask.sum():,}")

        self.phase= "sub_clustering"
        self.cluster_history.append(len(np.unique(all_labels)))

        return all_labels, results

    def check_stability(self):
        if len(self.cluster_history) < self.stability_window:
            return False
        return len(set(self.cluster_history[-self.stability_window:])) == 1

    def fit_classification(self, X, labels):
        X_s= self.scaler.transform(X)

        cv= StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

        candidates= {
            "random_forest": RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=5,
                class_weight="balanced",
                n_jobs=-1,
                random_state=SEED,)}

        if XGB_OK:
            candidates["xgboost"] = XGBClassifier(
                n_estimators= 300,
                learning_rate= 0.05,
                max_depth= 6,
                subsample= 0.8,
                colsample_bytree= 0.8,
                eval_metric= "mlogloss",
                random_state= SEED,
                n_jobs= -1,
            )

        if LGB_OK:
            candidates["lightgbm"]= LGBMClassifier(
                n_estimators= 300,
                learning_rate= 0.05,
                num_leaves= 31,
                min_child_samples= 20,
                class_weight= "balanced",
                random_state= SEED,
                n_jobs= -1,
                verbosity= -1,)

        best_score= 0.0
        best_name= ""
        best_model= None

        for name, model in candidates.items():
            sc= cross_val_score(
                model, X_s, labels,
                cv=cv, scoring="f1_macro", n_jobs=-1,)
            print(f"{name}: {sc.mean():.4f} ± {sc.std():.4f}")

            if sc.mean() > best_score:
                best_score= sc.mean()
                best_name= name
                best_model= model

        best_model.fit(X_s,labels)
        self.classifier= best_model
        self.classifier_name= best_name
        self.phase= "classification"

        print(f" → Selected: {best_name}  F1={best_score:.4f}")
        return best_name, best_score

    def predict(self, x_row, cpap_cols):
        x= np.array([float(x_row[c]) if c in x_row.index else 0.0
            for c in cpap_cols])
        macro= self._macro_rule(x, cpap_cols)

        if self.phase== "classification" and self.classifier is not None:
            X_s= self.scaler.transform(x.reshape(1, -1))
            label= int(self.classifier.predict(X_s)[0])
            proba= self.classifier.predict_proba(X_s)[0]
        else:
            label= macro
            proba= np.zeros(4)
            proba[macro]= 1.0

        return label, float(proba.max()), proba

    def _macro_rule(self, x, cpap_cols):
        col_idx= {c: i for i, c in enumerate(cpap_cols)}
        def _get(col_7d, col_old, fallback_val):
            """Return feature value: prefer 7d name, fall back to old name."""
            for c in (col_7d, col_old):
                if c in col_idx:
                    return x[col_idx[c]]
            return fallback_val

        use= _get("use_mean_7d","use_mean_x", 0.0)
        ahi= _get("ahi_mean_7d","ahi_mean_x", 10.0)
        slp= _get("use_slope","use_slope",  0.0)
        low= _get("low_use_streak", "low_use_streak", 0)

        if use < 2.0 or low >= 3:
            return 3
        if slp < -0.3 or low >= 2:
            return 2
        if use >= 6.0 and ahi <= 10 and low == 0:
            return 0
        return 1

class SurveySpecialist:
    CLASSES= {
        0: "well_controlled",
        1: "subclinical",
        2: "at_risk",
        3: "clinical_concern",}

    def __init__(self):
        self.model= None
        self.scaler= StandardScaler()
        self._cols= []

    def engineer_labels(self, df):
        """
        Assign severity class from clinical thresholds.
        Expects df to contain only patients WITH survey data
        (ESS_missing==0 or similar).  Calling this on patients
        with all scores NaN would label them as class 0 via fillna(0).
        The fit() method enforces this filter before calling here.
        """
        ess= df["ESS_score"].fillna(0)
        psqi= df["PSQI_score"].fillna(0)
        bdi= df["BDI_score"].fillna(0)
        isi= df["ISI_score"].fillna(0)
        fss= df["FSS_score"].fillna(0)

        abnorm= ((ess  >= 8) +
            (psqi >= 6) +(bdi  >= 11) +
            (isi  >= 8) +(fss  >= 4)).astype(int)

        severe= ((ess  >= 12)+
            (psqi >= 11)+(bdi  >= 20) +
            (isi  >= 15)+(fss  >= 5)).astype(int)

        labels= np.zeros(len(df), dtype=int)
        labels[abnorm >= 1]= 1
        labels[severe >= 1]= 2
        labels[(severe >= 2) | ((bdi >= 20) & (fss >= 5))] = 3
        return labels

    def fit(self, df):
        self._cols= [c for c in SURVEY_FEATS if c in df.columns]
        _miss_cols= [c for c in self._cols if c.endswith("_missing")]
        if _miss_cols:
            has_survey= (df[_miss_cols] == 0).any(axis=1)
            df_train= df[has_survey].copy()
        else:
            df_train= df.copy()

        if len(df_train)< 50:
            print(f"SurveySpecialist: only {len(df_train)} survey patients "
                  f"(< 50 minimum) — skipping fit.")
            return None

        X= df_train[self._cols].fillna(0)
        y= self.engineer_labels(df_train)
        X_s= self.scaler.fit_transform(X)

        if CAT_OK:
            self.model= CatBoostClassifier(
                iterations= 300,
                learning_rate= 0.05,
                depth= 6,
                loss_function= "MultiClass",
                auto_class_weights= "Balanced",
                verbose= 0,
                random_seed= SEED,)
        else:
            self.model= RandomForestClassifier(
                n_estimators= 200,
                class_weight= "balanced",
                random_state= SEED,
                n_jobs= -1,)

        self.model.fit(X_s, y)

        from collections import Counter
        print(f"SurveySpecialist fitted on {len(df_train):,} patients | "
              f"label dist:{dict(Counter(y))}")
        return y

    def predict(self, x_row):
        if self.model is None or not self._cols:
            return -1, 0.0, np.zeros(4)

        x= np.array([float(x_row[c]) if c in x_row.index else 0.0
                         for c in self._cols])
        X_s= self.scaler.transform(x.reshape(1, -1))

        label= int(self.model.predict(X_s)[0])
        proba= self.model.predict_proba(X_s)[0]

        return label, float(proba.max()), proba

class CareSpecialist:
    CLASSES= {
        0: "optimal_care",
        1: "adequate_care",
        2: "suboptimal_care",
        3: "problematic_care",}

    def __init__(self):
        self.model= None
        self.scaler= StandardScaler()
        self._cols= []

    def engineer_labels(self, df):
        n_int= df["intervention_count_30d"].fillna(0)
        pend= df["pending_count"].fillna(0)
        p_age= df["pending_max_age"].fillna(0)
        d_last= df["days_since_last"].fillna(999)
        labels= np.full(len(df), 2, dtype=int)

        labels[(n_int >= 1) & (d_last <= 30)] = 1
        labels[(n_int >= 1) & (d_last <= 14)] = 0
        labels[(p_age > 21) | (pend > 2)] = 3

        return labels

    def fit(self, df):
        self._cols= [c for c in CARE_FEATS if c in df.columns]
        X= df[self._cols].fillna(0)
        y= self.engineer_labels(df)
        X_s= self.scaler.fit_transform(X)

        from collections import Counter
        unique_classes = len(np.unique(y))
        zero_var = bool(np.all(np.std(X_s, axis=0) < 1e-6)) if len(X_s) > 0 else True

        if unique_classes <= 1 or zero_var:
            print(f"CareSpecialist: insufficient variance / single class ({unique_classes} classes) — using rule-based fallback | label dist: {dict(Counter(y))}")
            self.model = None
            return y

        try:
            if CAT_OK:
                self.model= CatBoostClassifier(
                    iterations= 300,
                    learning_rate= 0.05,
                    depth= 6,
                    loss_function= "MultiClass",
                    auto_class_weights= "Balanced",
                    verbose= 0,
                    random_seed= SEED,)
            else:
                self.model= RandomForestClassifier(
                    n_estimators= 200,
                    class_weight= "balanced",
                    random_state= SEED,
                    n_jobs= -1,)
            self.model.fit(X_s, y)
            print(f"CareSpecialist fitted on {len(df):,} patients | "f"label dist: {dict(Counter(y))}")
        except Exception as e:
            print(f"CareSpecialist fit notice: ({e}) — falling back to clinical rules.")
            self.model = None
        return y

    def predict(self, x_row):
        if self.model is None or not self._cols:
            d = pd.DataFrame([x_row])
            label = int(self.engineer_labels(d)[0])
            proba = np.zeros(4)
            proba[label] = 1.0
            return label, 0.85, proba

        x= np.array([
            float(x_row[c]) if c in x_row.index else 0.0
            for c in self._cols])
        X_s= self.scaler.transform(x.reshape(1, -1))

        label= int(self.model.predict(X_s)[0])
        proba= self.model.predict_proba(X_s)[0]

        return label, float(proba.max()), proba

class BiomarkerSpecialist:
    CLASSES= {
        0: "good_biomarker",
        1: "moderate_concern",
        2: "poor_biomarker",
        3: "critical_alert",}

    def __init__(self):
        self.model= None
        self.scaler= StandardScaler()
        self._feature_names= []

    def engineer_labels(self, df):
        """
        Expects df to contain ONLY patients with bio data (bio_absent == 0).
        fillna defaults are clinically neutral — they do not trigger any
        threshold — so they must not be applied to non-bio patients.
        fit() enforces this filter before calling here.
        """
        spo2= df["spo2_7d"].fillna(95)
        hrv= df["hrv_rmssd_7d"].fillna(30)
        sleep= df["sleep_efficiency_7d"].fillna(0.75)  
        score= df["sleep_score_7d"].fillna(65)
        afib= df["afib_detected"].fillna(0)
        sbp= df["systolic_bp_7d"].fillna(120)

        labels= np.zeros(len(df), dtype=int)            

        labels[(spo2 < 95) | (hrv < 20)  | (sleep < 0.65) | (score < 60)] = 1
        labels[(spo2 < 92) | (hrv < 12)  | (sleep < 0.55) | (sbp > 150)]= 2
        labels[(spo2 < 88) | (afib > 0)  | (sbp > 165)]= 3

        return labels

    def _modality_dropout(self, X, cols, rate=0.30, random_state=None):
        rng= (np.random.RandomState(random_state)
               if random_state is not None
               else np.random.RandomState())
        X_aug= X.copy()
        groups= {
            "ww":[i for i, c in enumerate(cols)
                   if c in ["hrv_rmssd_7d", "spo2_ww_7d",
                             "sleep_efficiency_7d", "sleep_score_7d",
                             "snoring_s_7d", "wakeup_count_7d"]],
            "masimo":[i for i, c in enumerate(cols)
                         if c in ["spo2_masimo_7d", "pvi_7d"]],
            "somnoart":[i for i, c in enumerate(cols)
                         if c in ["tst_min_7d", "waso_min_7d",
                                  "n3_duration_7d", "rem_duration_7d",
                                  "nb_awakenings_7d"]],
            "hexoskin":[i for i, c in enumerate(cols)
                         if c in ["hrv_lf_hf_ratio_7d", "breathing_rate_7d"]],
            "bpm":[i for i, c in enumerate(cols)
                         if c in ["systolic_bp_7d", "diastolic_bp_7d",
                                  "afib_detected"]],}

        absent= {d: cols.index(f"{d}_absent") if f"{d}_absent" in cols else None
            for d in ["ww", "masimo", "somnoart", "hexoskin", "bpm"]}

        for dev, idx in groups.items():
            mask= rng.random(len(X)) < rate
            if idx:X_aug[np.ix_(mask, idx)] = 0.0
            if absent.get(dev) is not None:X_aug[mask, absent[dev]] = 1.0
        return X_aug

    def fit(self, df):
        self._feature_names= [c for c in BIO_FEATS if c in df.columns]
        if "bio_absent" in df.columns:
            df_train= df[df["bio_absent"] == 0].copy()
        else:
            df_train= df.copy()

        if len(df_train) < 20:
            print(f"BiomarkerSpecialist: only {len(df_train)} bio patients "f"(< 20 minimum) — skipping fit.")
            return None

        X_orig= df_train[self._feature_names].fillna(0).values
        y_orig= self.engineer_labels(df_train)
        X_aug= self._modality_dropout(X_orig, self._feature_names,random_state=SEED)

        X_all= np.vstack([X_orig, X_aug])
        y_all = np.concatenate([y_orig, y_orig])
        self.scaler.fit(X_orig)
        X_s= self.scaler.transform(X_all)

        if LGB_OK:
            self.model= LGBMClassifier(
                n_estimators= 300,
                learning_rate= 0.05,
                num_leaves= 31,
                min_child_samples= 5,
                class_weight= "balanced",
                random_state= SEED,
                n_jobs= -1,
                verbosity= -1,)
        else:
            self.model= RandomForestClassifier(
                n_estimators= 200,
                class_weight= "balanced",
                random_state= SEED,
                n_jobs= -1,)
        self.model.fit(X_s, y_all)

        from collections import Counter
        print(f"BiomarkerSpecialist fitted on {len(df_train):,} patients | "f"label dist: {dict(Counter(y_orig))}")
        return y_orig

    def predict(self, x_row):
        if self.model is None or not self._feature_names:
            return -1, 0.0, np.zeros(4)

        x= np.array([float(x_row[c]) if c in x_row.index else 0.0
                         for c in self._feature_names])
        X_s= self.scaler.transform(x.reshape(1, -1))

        label= int(self.model.predict(X_s)[0])
        proba= self.model.predict_proba(X_s)[0]
        _all_absent= ["ww_absent", "masimo_absent", "somnoart_absent","hexoskin_absent", "bpm_absent"]
        n_abs= sum(1 for c in _all_absent if c in x_row.index and x_row[c] == 1)
        rel= float(proba.max()) * max(0.4, 1.0 - 0.1 * n_abs)
        return label, rel, proba

class DecisionMatrix:
    COMBINED_STATES= {
        0: "stable_controlled",
        1: "clinical_monitored",
        2: "clinical_attention",
        3: "clinical_urgent",
        4: "physician_referral",}
    ACTION_SCOPE = {
        0: "routine_monitoring",
        1: "proactive_outreach",
        2: "technician_visit",
        3: "urgent_visit",
        4: "physician_referral",}

    def dempster_shafer_combine(self, specialist_outputs):
        n= len(self.COMBINED_STATES)
        combined= np.ones(n)

        for out in specialist_outputs.values():
            if out is None:
                continue

            reliability= out["reliability"]
            proba= np.zeros(n)
            for i, v in enumerate(out["proba"][:n]):
                proba[i]= v
            proba/= proba.sum() + 1e-10
            mass= reliability * proba + (1 - reliability) / n
            combined*= mass

        total= combined.sum()
        if total > 0:combined /= total

        return int(np.argmax(combined)), combined

    def run(self, specialist_outputs):
        state, beliefs = self.dempster_shafer_combine(specialist_outputs)
        conflict= float(1 - beliefs.max())
        n_severe= sum(
            1 for out in specialist_outputs.values()
            if out is not None and out.get("label", -1) >= 3)
        if conflict>= 0.80 or (state == 3 and n_severe >= 2):
            state= min(state + 1, len(self.COMBINED_STATES) - 1)

        return {
            "combined_state":state,
            "state_name":self.COMBINED_STATES[state],
            "action_scope": self.ACTION_SCOPE[state],
            "conflict_score":round(conflict, 4),
            "physician_trigger":int(state >= 4),
            "urgent_flag":int(state >= 3),}

## LAYER 2 — TRAINING

In [31]:
#INSTANTIATE SPECIALISTS

cpap_sp= CPAPSpecialist()
surv_sp= SurveySpecialist()
care_sp= CareSpecialist()
bio_sp= BiomarkerSpecialist()
dm= DecisionMatrix()

#CPAP SPECIALIST — LEVEL 1: MACRO STATES

print("\nCPAP Specialist — Level 1 macro states:")
macro_labels = cpap_sp.assign_macro_state(features)
features["cpap_macro_state"]= macro_labels

for k, name in CPAPSpecialist.MACRO_STATES.items():
    n= (macro_labels== k).sum()
    print(f"{k} {name:<25}: {n:,}({n/len(features)*100:.1f}%)")

#CPAP SPECIALIST — LEVEL 2: MEAN SHIFT SUB-CLUSTERS

print("\nCPAP Specialist — Level 2 Mean Shift sub-clusters:")
sub_labels, cpap_cluster_results = cpap_sp.fit_clustering(features,macro_labels,)
features["cpap_cluster"]= sub_labels

#CARE SPECIALIST

print("\nCare Specialist:")
care_labels= care_sp.fit(features)
features["care_state"]= care_labels
for k, name in CareSpecialist.CLASSES.items():
    n = (care_labels== k).sum()
    print(f"{k}{name:<20}: {n:,} ({n / len(features)* 100:.1f}%)")


CPAP Specialist — Level 1 macro states:
0 stable_adherent          : 16,436(40.0%)
1 moderate_user            : 13,880(33.8%)
2 declining_user           : 7,090(17.2%)
3 critical_non_adherent    : 3,709(9.0%)

CPAP Specialist — Level 2 Mean Shift sub-clusters:
  State 1 (moderate_user): K_sub=32  sil=0.420  n=13,880
  State 2 (declining_user): K_sub=21  sil=0.396  n=7,090
  State 3 (critical_non_adherent): K_sub=17  sil=0.397  n=3,709

Care Specialist:
CareSpecialist fitted on 41,115 patients | label dist: {1: 3999, 3: 1309, 2: 30288, 0: 5519}
0optimal_care        : 5,519 (13.4%)
1adequate_care       : 3,999 (9.7%)
2suboptimal_care     : 30,288 (73.7%)
3problematic_care    : 1,309 (3.2%)


=## LAYER 2 — CLUSTER EXPORT

In [33]:
cpap_layer2= features[["AtHomePatientId", "cpap_macro_state", "cpap_cluster"]].copy()
cpap_layer2["macro_state_name"]= cpap_layer2["cpap_macro_state"].map(CPAPSpecialist.MACRO_STATES)
cpap_layer2.to_csv("cpap_layer2_clusters.csv", index=False)
print(f"Exported: cpap_layer2_clusters.csv | "f"{len(cpap_layer2):,} rows × {cpap_layer2.shape[1]} columns")

#CPAP CLUSTERING SUMMARY

summary_rows= []
n_state0= (macro_labels== 0).sum()
summary_rows.append({"macro_state":0,"macro_state_name": CPAPSpecialist.MACRO_STATES[0],
    "K_sub":1,
    "silhouette":None,
    "n_patients":int(n_state0),})

for state, result in cpap_cluster_results.items():
    summary_rows.append({"macro_state":state,
        "macro_state_name": CPAPSpecialist.MACRO_STATES.get(state, ""),
        "K_sub":result["K"],
        "silhouette":round(result["silhouette"], 4),
        "n_patients":result["n"],})

summary_df= pd.DataFrame(summary_rows).sort_values("macro_state").reset_index(drop=True)
summary_df.to_csv("cpap_clustering_summary.csv", index=False)
print(f"Exported: cpap_clustering_summary.csv|{len(summary_df)} states")
print(summary_df.to_string(index=False))

Exported: cpap_layer2_clusters.csv | 41,115 rows × 4 columns
Exported: cpap_clustering_summary.csv|4 states
 macro_state      macro_state_name  K_sub  silhouette  n_patients
           0       stable_adherent      1         NaN       16436
           1         moderate_user     32      0.4204       13880
           2        declining_user     21      0.3959        7090
           3 critical_non_adherent     17      0.3972        3709


## LAYER 2 — SURVEY SPECIALIST TRAINING + EXPORT

In [34]:
survey_missing_cols= [
    c for c in ["ESS_missing", "PSQI_missing", "BDI_missing", "ISI_missing","FSS_missing", "SF36_missing"]
    if c in features.columns]
surv_mask= (features[survey_missing_cols] == 0).any(axis=1)
X_surv= features[surv_mask]

if len(X_surv)>= 50:
    surv_labels= surv_sp.fit(X_surv)
    if surv_labels is None:
        features["survey_state"] = -1
        print("Not enough valid survey records after internal filter.")
    else:
        features.loc[surv_mask, "survey_state"]= surv_labels
        features["survey_state"]= features["survey_state"].fillna(-1).astype(int)
        for k, n in zip(*np.unique(surv_labels, return_counts=True)):
            print(f"{k} {SurveySpecialist.CLASSES[k]:<22}: {n:,}")
else:
    features["survey_state"]= -1
    print(f" Not enough survey records ({len(X_surv)}) for model training.")
#SURVEY LAYER 2 — PATIENT-LEVEL STATE ASSIGNMENTS
if "survey_state" not in features.columns:
    features["survey_state"]= -1
    print(" Warning: survey_state not found — defaulting to -1 (no_survey_data).")
    print(" Re-run the Survey Specialist training block first.")

survey_layer2= features[["AtHomePatientId", "survey_state"]].copy()
survey_layer2["survey_state_name"] = survey_layer2["survey_state"].map(
    {**SurveySpecialist.CLASSES, -1: "no_survey_data"})

survey_layer2.to_csv("survey_layer2_states.csv", index=False)
print(f"Exported: survey_layer2_states.csv| "f"{len(survey_layer2):,} rows × {survey_layer2.shape[1]} columns")

print("\nState distribution:")
print(survey_layer2["survey_state_name"].value_counts().to_string())

SurveySpecialist fitted on 4,837 patients | label dist:{0: 3857, 1: 537, 2: 381, 3: 62}
0 well_controlled       : 3,857
1 subclinical           : 537
2 at_risk               : 381
3 clinical_concern      : 62
Exported: survey_layer2_states.csv| 41,115 rows × 3 columns

State distribution:
survey_state_name
no_survey_data      36278
well_controlled      3857
subclinical           537
at_risk               381
clinical_concern       62


## LAYER 2 — BIO SPECIALIST TRAINING + EXPORT

In [35]:
if "in_poc" not in features.columns or "bio_absent" not in features.columns:
    features["bio_state"]= -1
    print(" Warning: in_poc / bio_absent not in features — ""re-run the features merge block first.")
else:
    bio_mask= features["in_poc"] & (features["bio_absent"] == 0)
    X_bio= features[bio_mask]

    if len(X_bio)>= 50:
        bio_labels= bio_sp.fit(X_bio)
        if bio_labels is None:
            features["bio_state"] = -1
            print("Not enough valid biomarker records after internal filter.")
        else:
            features.loc[bio_mask, "bio_state"] = bio_labels
            features["bio_state"]= features["bio_state"].fillna(-1).astype(int)
            for k, n in zip(*np.unique(bio_labels, return_counts=True)):
                print(f" {k} {BiomarkerSpecialist.CLASSES[k]:<22}: {n:,}")
    else:
        features["bio_state"]= -1
        print(f"  Not enough biomarker records ({len(X_bio)}) for model training.")
#BIOMARKER LAYER 2 — PATIENT-LEVEL STATE ASSIGNMENTS
if "bio_state" not in features.columns:
    features["bio_state"] = -1
    print(" Warning: bio_state not found — defaulting to -1 (no_bio_data).")
    print(" Re-run the Biomarker Specialist training block first.")

bio_layer2= features[["AtHomePatientId", "bio_state", "in_poc"]].copy()
bio_layer2["bio_state_name"]= bio_layer2["bio_state"].map(
    {**BiomarkerSpecialist.CLASSES, -1: "no_bio_data"})

bio_layer2.to_csv("bio_layer2_states.csv", index=False)
print(f"Exported: bio_layer2_states.csv |  "f"{len(bio_layer2):,} rows × {bio_layer2.shape[1]} columns")

print("\nState distribution:")
print(bio_layer2["bio_state_name"].value_counts().to_string())

BiomarkerSpecialist fitted on 396 patients | label dist: {1: 90, 3: 148, 2: 138, 0: 20}
 0 good_biomarker        : 20
 1 moderate_concern      : 90
 2 poor_biomarker        : 138
 3 critical_alert        : 148
Exported: bio_layer2_states.csv |  41,115 rows × 4 columns

State distribution:
bio_state_name
no_bio_data         40719
critical_alert        148
poor_biomarker        138
moderate_concern       90
good_biomarker         20


## LAYER 2 — DS FUSION

In [37]:
def build_outputs(row):
    out= {}
    # CPAP — always present
    m= int(row["cpap_macro_state"])
    p= np.zeros(5)
    p[min(m, 4)]= 1.0
    out["cpap"]= {"label":m,"reliability": float(row.get("phi_cpap", 1.0)),"proba":p,}
    # Survey — only when survey_state >= 0
    if row.get("survey_state", -1)>= 0:
        s= int(row["survey_state"])
        p= np.zeros(5)
        p[min(s, 4)]= 1.0
        out["surv"]= {"label":s,"reliability": float(row.get("phi_surv", 0.3)),"proba":p,}

    # Care — only when care_state >= 0
    if row.get("care_state", -1) >= 0:
        c= int(row["care_state"])
        p= np.zeros(5)
        p[min(c, 4)]= 1.0
        out["care"]= {"label":c,"reliability": float(row.get("phi_care", 0.5)),"proba":p,}
    # Bio — only when bio_state >= 0 (PoC patients with device data)
    if row.get("bio_state", -1) >= 0:
        b= int(row["bio_state"])
        p= np.zeros(5)
        p[min(b, 4)]= 1.0
        out["bio"]= {"label":b,"reliability": float(row.get("phi_bio", 0.3)),"proba":p,}
    return out

_dm_cols= ["combined_state", "state_name", "action_scope","conflict_score", "physician_trigger", "urgent_flag"]
features.drop(columns=[c for c in _dm_cols if c in features.columns],inplace=True)

dm_out= features.apply(lambda r: pd.Series(dm.run(build_outputs(r))), axis=1)
features= pd.concat([features, dm_out], axis=1)

## LAYER 2 — SUMMARY DISPLAY

In [38]:
for k, name in CPAPSpecialist.MACRO_STATES.items():
    n= (features["cpap_macro_state"]== k).sum()
    print(f"{k} {name:<25}: {n:>7,} ({n / len(features) * 100:.1f}%)")

print("\nCombined state distribution:")
for k, name in DecisionMatrix.COMBINED_STATES.items():
    n= (features["combined_state"] == k).sum()
    print(f"{k} {name:<22}: {n:>7,}")

print(f"\nPhysician triggers:{features['physician_trigger'].sum():,}")
print(f"Urgent flags: {features['urgent_flag'].sum():,}")

poc_s= features[features["in_poc"]].head(5)

use_col=("use_mean_7d" if "use_mean_7d" in poc_s.columns
           else "use_mean_x" if "use_mean_x" in poc_s.columns
           else "use_mean")

sample_cols= [
    "AtHomePatientId",
    use_col,"cpap_macro_state","survey_state",
    "care_state","bio_state","combined_state",
    "state_name","action_scope","conflict_score",]
sample_cols= [c for c in sample_cols if c in poc_s.columns]

print("\nSample — 5 PoC patients:")
print(poc_s[sample_cols].to_string(index=False))

0 stable_adherent          :  16,436 (40.0%)
1 moderate_user            :  13,880 (33.8%)
2 declining_user           :   7,090 (17.2%)
3 critical_non_adherent    :   3,709 (9.0%)

Combined state distribution:
0 stable_controlled     :  19,828
1 clinical_monitored    :  12,146
2 clinical_attention    :   6,203
3 clinical_urgent       :   2,763
4 physician_referral    :     175

Physician triggers:175
Urgent flags: 2,938

Sample — 5 PoC patients:
 AtHomePatientId  use_mean_7d  cpap_macro_state  survey_state  care_state  bio_state  combined_state         state_name       action_scope  conflict_score
             256       5.4225                 1             1           2          1               1 clinical_monitored proactive_outreach             0.0
            2297      10.0025                 1             2           2          1               2 clinical_attention   technician_visit             0.0
            2737       7.8650                 1             3           2          3  

## LAYER 3 — IMPORTS & SETUP

In [39]:
try:
    import lightgbm as lgb
    LGB_OK= True
except ImportError:
    LGB_OK= False

try:
    from xgboost import XGBRegressor
    XGB_OK= True
except ImportError:
    XGB_OK= False

try:
    from lifelines import CoxPHFitter
    COX_OK= True
except ImportError:
    COX_OK= False

try:
    from imblearn.ensemble import BalancedRandomForestClassifier
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    IMBLEARN_OK = True
except ImportError:
    IMBLEARN_OK = False
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

SEED= 42

_max_last= bl["last_date"].max()
_today= pd.Timestamp.now().normalize()
REF= min(_max_last, _today)

print(f"Layer 3 setup — REF: {REF.date()}")
print(f"(bl last_date max: {_max_last.date()}| today: {_today.date()})")
print(f"LGB: {LGB_OK} |XGB: {XGB_OK}|Cox: {COX_OK} | imblearn:{IMBLEARN_OK}")

Layer 3 setup — REF: 2025-12-31
(bl last_date max: 2025-12-31| today: 2026-07-16)
LGB: False |XGB: True|Cox: True | imblearn:True


## LAYER 3 — CANCELLATION TAXONOMY + LABEL PREPARATION

In [40]:
NATURAL_REASONS = [
    "Intolérance / Inobservance / Pas d'amélioration",   # intolerance / non-compliance
    "Z### NPU #### Arrêt contre décharge",                # stopped against medical advice
    "IU refusée par le patient pour raisons perso",       # refused for personal reasons
    "Problème de prise en charge - patient injoignable",  # unreachable = disengaged
    "Patient refuse rdv médecin pour sa prise en charge", # refuses doctor appointment
    "Z### NPU #### Refus d'installation",                 # refused installation
    "Z### NPU #### Le patient refuse l'installation",
    "Prescription échue",
    "Patient mécontent - Changement de prestataire",]

#EFFECTFUL: care-process or physician-triggered ending
EFFECTFUL_REASONS = [
    "Demande du médecin",
    "Z### NPU #### Demande prescripteur",
    "Médecin mécontent - Changement de prestataire",
    "Z### NPU #### Changement de prestataire",
    "Changement de médecin - ne travaille pas avec LHF",
    "IU annulée par le médecin pour raisons médicales",]

EXCLUDED_REASONS = [
    "Changement de traitement (Switch code Thérapie)",
    "Changement de traitement (Hors Switch)",
    # Natural end of life/ clinical success — not behavioral dropout
    "Décès du patient",                                   #death
    "Z### NPU #### Fin de traitement",                    #treatment completed
    "Amélioration clinique",                              #clinical improvement
    # Logistical/ administrative — not behavioral
    "Déménagement hors zone LHF",                         #moved out of zone
    "Transfert CO",                                       #transferred to another org
    "Hospitalisation",                                    #hospitalization (pause)
    "Transfert vacances (Annulation temporaire)",          #temporary transfer
    "Erreur de saisie",                                   #data entry error
    "Z### NPU #### Annulation document hors delais",      #admin cancellation
    "Z### NPU #### Installation annulée",                 #installation canceled
    "Z### NPU #### Début de transfert vacances",          #vacation transfer start
    "Fin de transfert vacances",                          #vacation transfer end
    "Maison de soins assurant la prestation",             # care home takes over
    "IU annulée car patient équipé par un autre PSAD",    #already equipped elsewhere
]

def _check_taxonomy_coverage(cancellation_series):
    all_classified = set(NATURAL_REASONS) | set(EFFECTFUL_REASONS) | set(EXCLUDED_REASONS)
    s= cancellation_series.dropna()
    n= s.isin(NATURAL_REASONS).sum()
    e= s.isin(EFFECTFUL_REASONS).sum()
    ex= s.isin(EXCLUDED_REASONS).sum()
    uc= (~s.isin(all_classified)).sum()
    total= len(s)
    print(f"Natural:{n:>7,}({n/total*100:.1f}%)")
    print(f"Effectful:{e:>7,}({e/total*100:.1f}%)")
    print(f"Excluded:{ex:>7,}({ex/total*100:.1f}%)")
    print(f"Uncovered: {uc:>7,}({uc/total*100:.1f}%)")
    if uc > 0:
        print("Uncovered reasons:")
        print(s[~s.isin(all_classified)].value_counts().to_string())

startdate= pd.read_csv("all startdate.csv",engine = "python",on_bad_lines = "skip",)
startdate["AtHomePatientId"]= startdate["AtHomePatientId"].astype(int)
startdate["TherapyStartDate"]= pd.to_datetime(startdate["TherapyStartDate"], errors="coerce")
startdate["TherapyEndDate"]= pd.to_datetime(startdate["TherapyEndDate"],   errors="coerce")

print(f"Loaded all_startdate.csv: {len(startdate):,} rows")

print("Cancellation-reason taxonomy coverage check:")
_check_taxonomy_coverage(startdate["CancellationReason"])

excl_mask= startdate["CancellationReason"].isin(EXCLUDED_REASONS)
sd_clean= startdate[~excl_mask].copy()

print(f"Excluded (non-dropout events): {excl_mask.sum():,} "f"({excl_mask.mean()*100:.1f}%)")
print(f"Remaining for dropout analysis: {len(sd_clean):,}")

#NATURAL / EFFECTFUL LABEL ASSIGNMENT

sd_clean["y_natural"]= sd_clean["CancellationReason"].isin(NATURAL_REASONS).astype(int)
sd_clean["y_effectful"]= sd_clean["CancellationReason"].isin(EFFECTFUL_REASONS).astype(int)

print(f"\nNatural dropouts:{sd_clean['y_natural'].sum():,} "f"({sd_clean['y_natural'].mean()*100:.1f}%)")
print(f"Effectful dropouts: {sd_clean['y_effectful'].sum():,} "f"({sd_clean['y_effectful'].mean()*100:.1f}%)")
sd_clean["is_natural_ep"]= sd_clean["CancellationReason"].isin(NATURAL_REASONS).astype(int)
sd_clean["is_effectful_ep"]= sd_clean["CancellationReason"].isin(EFFECTFUL_REASONS).astype(int)

sd_clean["_max_start"]= (sd_clean.groupby("AtHomePatientId")["TherapyStartDate"].transform("max"))
sd_clean["has_restart"]= (sd_clean["TherapyEndDate"].notna() &(sd_clean["TherapyEndDate"] < sd_clean["_max_start"]))

sd_clean= sd_clean.drop(columns=["_max_start"])

sd_last= (sd_clean.sort_values("TherapyStartDate").groupby("AtHomePatientId", as_index=False).tail(1).reset_index(drop=True))

true_ended= sd_last["RenderedTreatmentStatus"] == "B"

sd_last["true_natural"] = (true_ended& ~sd_last["has_restart"].astype(bool)& (sd_last["is_natural_ep"] == 1)).astype(int)

sd_last["true_effectful"]= (true_ended& ~sd_last["has_restart"].astype(bool)& (sd_last["is_effectful_ep"] == 1)).astype(int)

sd_last["is_active"]= (sd_last["RenderedTreatmentStatus"] == "A").astype(int)

print("Label summary (all patients in startdate):")
print(f"True natural dropout:{sd_last['true_natural'].sum():,}")
print(f"True effectful dropout:{sd_last['true_effectful'].sum():,}")
print(f"Active (censored):{sd_last['is_active'].sum():,}")
print(f"Both labels= 1 (should be 0): "f"{((sd_last['true_natural']==1) & (sd_last['true_effectful']==1)).sum()}")

Loaded all_startdate.csv: 168,924 rows
Cancellation-reason taxonomy coverage check:
Natural:  9,932(7.9%)
Effectful: 10,779(8.6%)
Excluded: 96,090(76.4%)
Uncovered:   8,929(7.1%)
Uncovered reasons:
CancellationReason
Z### NPU #### Autre: à préciser                       8352
Z### NPU #### Autres                                   405
Z### NPU #### Orthèse                                  112
Z### NPU #### Contraintes légales/internes              22
Echantillon commercial                                  15
Z### NPU #### Opération médicale                        11
Z### NPU #### Fenêtre Thérapeutique                      6
Problème de prise en charge -patient sans mutuelle       3
Z### NPU #### Mauvais payeur                             2
Z### NPU #### Migration calea                            1
Excluded (non-dropout events): 96,090 (56.9%)
Remaining for dropout analysis: 72,834

Natural dropouts:9,932 (13.6%)
Effectful dropouts: 10,779 (14.8%)
Label summary (all patients in startdate)

## LAYER 3 — DURATION FOR COX PH

In [41]:
sd_last["end_for_dur"]= sd_last["TherapyEndDate"].clip(upper=REF)
sd_last["duration_days"]= (
    sd_last["end_for_dur"]- sd_last["TherapyStartDate"]).dt.days

sd_last["duration_cox"]= sd_last["duration_days"].clip(upper=3650)

sd_cox= sd_last[sd_last["duration_days"].notna() &(sd_last["duration_days"] > 0)].copy()

print("\nCox PH base:")
print(f"Total rows:{len(sd_cox):,}")
print(f" Nat events:{sd_cox['true_natural'].sum():,} "f"mean={sd_cox[sd_cox['true_natural']==1]['duration_cox'].mean():.0f}d")
print(f" Eff events:{sd_cox['true_effectful'].sum():,} "f"mean={sd_cox[sd_cox['true_effectful']==1]['duration_cox'].mean():.0f}d")
print(f" Active (censored at ≤3650d): {sd_cox['is_active'].sum():,}")
print(f"\n duration_cox: min={sd_cox['duration_cox'].min():.0f}d "f"median={sd_cox['duration_cox'].median():.0f}d "f"max={sd_cox['duration_cox'].max():.0f}d")


Cox PH base:
Total rows:26,341
 Nat events:9,299 mean=928d
 Eff events:10,153 mean=895d
 Active (censored at ≤3650d): 0

 duration_cox: min=1d median=631d max=3650d


## LAYER 3 — FEATURE LISTS (DROPOUT_FEATS / COX_FEATS)

In [42]:
DROPOUT_FEATS=[
    "use_mean_7d",
    "use_var",
    "use_slope",
    "ahi_mean_7d",
    "ahi_var",
    "leaks95_mean_7d",     # alias → leaks_high_7d
    "leaks_high_7d",       # % nights above device threshold (ML primary)
    "leaks_severity_7d",   # ordinal leak severity 0-2
    "leaks_zscore_dev",    # z-score vs device peers
    "leaks_worst_7d",      # peak leak in 7d
    "leaks_var_7d",        # nightly leak variability
    "low_use_streak",
    "instability_idx",
    "use_zscore",
    "ahi_zscore",
    "transmission_gap",
    "nights_valid",
    "pressure_mean",
    "pressure_missing",
    "intervention_count_30d",
    "intervention_count_90d",
    "intervention_diversity",
    "days_since_last",
    "pending_count",
    "pending_max_age",
    "visit_count_30d",
    "call_count_30d",
    "sms_count_30d",
    "last_intervention_t1",
    "last_intervention_t2",
    "last_intervention_t3",
    "response_rate",
    "care_missing",
    "ESS_score",   "ESS_missing",
    "PSQI_score",  "PSQI_missing",
    "BDI_score",   "BDI_missing",
    "ISI_score",   "ISI_missing",
    "hrv_rmssd_7d",
    "spo2_7d",
    "sleep_efficiency_7d",
    "bio_absent",
    "cpap_macro_state",
    "combined_state",
    "rho",]

COX_FEATS=[
    "use_mean_7d",
    "use_slope",
    "use_zscore",
    "instability_idx",
    "low_use_streak",
    "ahi_mean_7d",
    "leaks_high_7d",      # device-normalised leak feature
    "intervention_count_30d",
    "days_since_last",
    "ESS_score",
    "PSQI_score",
    "cpap_macro_state",]

# LAYER 3 — CLASSES (NaturalDropoutDetector, EffectfulDropoutDetector,TimeToDropoutModel)

class NaturalDropoutDetector:

    def __init__(self):
        self.model= None
        self.calibrator= None
        self.scaler= StandardScaler()
        self._cols= []
        self.alpha= 0.85
        self.beta= 0.15

    def asymmetric_loss(self, y_pred, data):
        y_true= data.get_label()
        p= np.clip(1.0 / (1.0 + np.exp(-y_pred)), 1e-7, 1 - 1e-7)
        grad= np.where(y_true == 1,-self.alpha * (1.0 - p),  self.beta * p)
        hess= np.where(y_true == 1, self.alpha * p * (1.0-p), self.beta * p * (1.0-p))
        return grad.astype(np.float64), hess.astype(np.float64)

    def fit(self, X, y):
        self._cols= list(X.columns)
        X_arr= self.scaler.fit_transform(X.fillna(0).values)
        if len(np.unique(y.values)) <= 1 or np.sum(y.values) < 5:
            print(f"NaturalDropoutDetector: insufficient positive labels ({np.sum(y.values)} events) — using baseline fallback.")
            self.model = None
            self.calibrator = None
            return

        X_tr, X_va, y_tr, y_va= train_test_split(
            X_arr, y.values, test_size=0.2, stratify=y.values, random_state=SEED)

        non_constant = np.std(X_tr, axis=0) > 1e-6
        if np.sum(non_constant) == 0:
            print("NaturalDropoutDetector: all features are constant — using baseline fallback.")
            self.model = None
            self.calibrator = None
            return

        if LGB_OK:
            try:
                dtrain= lgb.Dataset(X_tr, label=y_tr)
                dval= lgb.Dataset(X_va, label=y_va, reference=dtrain)
                params= {
                    "objective":self.asymmetric_loss,
                    "metric":["auc"],
                    "num_leaves":31,
                    "learning_rate":0.03,
                    "min_child_samples":20,
                    "subsample":0.8,
                    "colsample_bytree":0.8,
                    "seed":SEED,
                    "verbosity":-1,}
                self.model= lgb.train(
                    params,dtrain,
                    num_boost_round=1000,
                    valid_sets=[dval],
                    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)],)
                raw_va= self.model.predict(X_va)
            except Exception as e:
                print(f"NaturalDropoutDetector LightGBM notice ({e}) — falling back to RandomForest.")
                self.model= RandomForestClassifier(
                    n_estimators=300, class_weight="balanced",
                    random_state=SEED, n_jobs=-1,)
                self.model.fit(X_tr, y_tr)
                raw_va= self.model.predict_proba(X_va)[:, 1]
        else:
            self.model= RandomForestClassifier(
                n_estimators=300, class_weight="balanced",
                random_state=SEED, n_jobs=-1,)
            self.model.fit(X_tr, y_tr)
            raw_va= self.model.predict_proba(X_va)[:, 1]

        try:
            self.calibrator= LogisticRegression(C=1.0, max_iter=500)
            self.calibrator.fit(raw_va.reshape(-1, 1), y_va)
        except Exception:
            self.calibrator= None

    def predict_risk(self, X):
        if self.model is None or not self._cols:
            if isinstance(X, pd.DataFrame):
                return np.full(len(X), 0.05)
            return np.array([0.05])
        X_arr= self.scaler.transform(
            X[self._cols].fillna(0).values if isinstance(X, pd.DataFrame)
            else np.nan_to_num(X).reshape(1, -1))
        if hasattr(self.model, "predict") and LGB_OK and not isinstance(self.model, RandomForestClassifier):
            raw= self.model.predict(X_arr)
        else:
            raw= self.model.predict_proba(X_arr)[:, 1]
        if self.calibrator is not None:
            return self.calibrator.predict_proba(raw.reshape(-1, 1))[:, 1]
        return np.clip(raw, 0.0, 1.0)

class EffectfulDropoutDetector:

    def __init__(self):
        self.model= None
        self.scaler= StandardScaler()
        self._cols= []

    def fit(self, X, y_effectful):
        self._cols= list(X.columns)
        X_s= self.scaler.fit_transform(X.fillna(0).values)
        print(f"Effectful labels: {y_effectful.sum():,}/{len(y_effectful):,} "f"({y_effectful.mean()*100:.1f}%)")
        n_min= int(y_effectful.sum())
        n_maj= int((y_effectful == 0).sum())
        if n_min == 0:
            print("EffectfulDropoutDetector: 0 effectful labels — skipping model fit.")
            self.model = None
            return
        if IMBLEARN_OK and n_min >= 10 and n_maj >= 10:
            target_ratio = min(0.1, n_min / n_maj * 3)
            smote = SMOTE(k_neighbors = max(1, min(5, n_min - 1)),sampling_strategy = max(0.01, target_ratio),random_state = SEED,)
            self.model= ImbPipeline([
                ("smote", smote),
                ("brf", BalancedRandomForestClassifier(
                    n_estimators=300, min_samples_leaf=3,
                    sampling_strategy="auto", replacement=True,
                    class_weight="balanced_subsample",
                    n_jobs=-1, random_state=SEED,)),])
        else:
            self.model= RandomForestClassifier(
                n_estimators=200, class_weight="balanced",
                random_state=SEED, n_jobs=-1,)
        try:
            self.model.fit(X_s, y_effectful.values)
        except Exception as e:
            print(f"EffectfulDropoutDetector fit notice: ({e}) — model disabled.")
            self.model = None

    def predict_effectful(self, X):
        if self.model is None:
            return np.zeros(len(X))
        X_s= self.scaler.transform(
            X[self._cols].fillna(0).values if isinstance(X, pd.DataFrame)
            else np.nan_to_num(X).reshape(1, -1))
        p= self.model.predict_proba(X_s)
        return p[:, 1] if p.shape[1] > 1 else np.zeros(len(X_s))

class TimeToDropoutModel:

    def __init__(self):
        self.xgb_model= None
        self.cox_model= None
        self.scaler= StandardScaler()
        self._cols= []
        self._cox_cols= []

    def fit_xgb(self, X, y_days):
        self._cols= list(X.columns)
        X_s= self.scaler.fit_transform(X.fillna(0).values)
        X_tr, X_va, y_tr, y_va = train_test_split(
            X_s, y_days.values, test_size=0.2, random_state=SEED)
        if XGB_OK:
            try:
                self.xgb_model= XGBRegressor(
                    early_stopping_rounds= 50,
                    n_estimators= 500,
                    learning_rate= 0.03,
                    max_depth= 6,
                    subsample= 0.8,
                    colsample_bytree= 0.8,
                    objective= "reg:squarederror",
                    reg_alpha= 0.1,
                    reg_lambda= 1.0,
                    random_state= SEED,
                    n_jobs= -1,)
                self.xgb_model.fit(X_tr, y_tr,eval_set=[(X_va, y_va)], verbose=False)
            except TypeError:
                self.xgb_model= XGBRegressor(
                    n_estimators=500, learning_rate=0.03, max_depth=6,
                    subsample=0.8, colsample_bytree=0.8,
                    objective="reg:squarederror",
                    reg_alpha=0.1, reg_lambda=1.0,
                    random_state=SEED, n_jobs=-1,)
                self.xgb_model.fit(X_tr, y_tr,eval_set=[(X_va, y_va)], early_stopping_rounds=50, verbose=False)
            mae= float(np.mean(np.abs(self.xgb_model.predict(X_va) - y_va)))
            print(f"XGBoost TTD: MAE={mae:.1f} days  n={len(X_tr):,}")
        else:
            from sklearn.ensemble import GradientBoostingRegressor
            self.xgb_model= GradientBoostingRegressor(
                n_estimators=200, learning_rate=0.05,
                max_depth=4, random_state=SEED,)
            self.xgb_model.fit(X_tr, y_tr)

    def fit_cox(self, survival_df):
        if not COX_OK:
            return
        cox_feats= getattr(self, "_cox_feats_override", None) or COX_FEATS
        available= [c for c in cox_feats + ["duration", "event"]
                     if c in survival_df.columns]
        cox_df= survival_df[available].dropna()
        cox_df= cox_df[(cox_df["duration"] > 0) & (cox_df["duration"] < 3650)]
        if len(cox_df) < 100 or cox_df["event"].sum() < 10:
            print(f" Cox PH: insufficient ({len(cox_df)} rows, "f"{cox_df['event'].sum()} events)")
            return
        self._cox_cols= [c for c in available if c not in ["duration", "event"]]
        self.cox_model= CoxPHFitter(penalizer=1.0)
        self.cox_model.fit(cox_df, duration_col="duration",event_col="event", show_progress=False)
        try:
            c= self.cox_model.concordance_index_
        except Exception:
            c= float("nan")
        print(f" Cox PH: C-index={c:.3f}  n={len(cox_df):,} "f"events={int(cox_df['event'].sum()):,}")

    def predict(self, X_row, therapy_days=180.0):
        h_days= 365.0
        hazard= 0.001
        if self.xgb_model is not None:
            x = np.nan_to_num(np.array([X_row.get(c, 0.0) for c in self._cols]),nan=0.0)
            h_days= float(max(1, self.xgb_model.predict(
                self.scaler.transform(x.reshape(1, -1)))[0]))
        if self.cox_model is not None and self._cox_cols:
            try:
                X_cox= pd.DataFrame(
                    [{c: X_row.get(c, 0) for c in self._cox_cols}])
                sf= self.cox_model.predict_survival_function(X_cox)
                s= sf.iloc[:, 0]
                idx= min(np.searchsorted(s.index.values, therapy_days), len(s) - 2)
                s_t, s_t1= float(s.iloc[idx]), float(s.iloc[idx + 1])
                hazard= float(max(1e-4,-(np.log(s_t1 + 1e-9) - np.log(s_t + 1e-9)))) if s_t > 0 else 0.001
            except Exception:
                pass
        return h_days, hazard

## LAYER 3 — ANY-HISTORICAL DROPOUT LABELS + df_train ASSEMBLY

In [43]:
any_nat= (sd_clean[sd_clean["is_natural_ep"] == 1].groupby("AtHomePatientId")["TherapyEndDate"].max().reset_index())
any_nat.columns= ["AtHomePatientId", "last_nat_end"]
any_nat["y_natural"]= 1

any_eff= (sd_clean[sd_clean["is_effectful_ep"] == 1].groupby("AtHomePatientId")["TherapyEndDate"]
    .max().reset_index())
any_eff.columns= ["AtHomePatientId", "last_eff_end"]
any_eff["y_effectful"]= 1

print(f"Any-historical natural dropout:{len(any_nat):,} patients")
print(f"Any-historical effectful dropout:{len(any_eff):,} patients")
# Therapy duration from the most recent episode per patient
therapy_dur= sd_last[
    ["AtHomePatientId", "duration_days", "TherapyStartDate", "is_active"]].copy()
# Feature columns available in the current features DataFrame
feat_cols = [c for c in DROPOUT_FEATS if c in features.columns]

df_train= features[["AtHomePatientId"] + feat_cols].copy()

df_train= df_train.merge(sd_last[["AtHomePatientId", "true_natural"]].rename(columns={"true_natural": "y_natural"}),on="AtHomePatientId", how="left",)
df_train= df_train.merge(sd_last[["AtHomePatientId", "true_effectful"]].rename(columns={"true_effectful": "y_effectful"}),on="AtHomePatientId", how="left",)
df_train= df_train.merge(therapy_dur, on="AtHomePatientId", how="left")

df_train["y_natural"]= df_train["y_natural"].fillna(0).astype(int)
df_train["y_effectful"]= df_train["y_effectful"].fillna(0).astype(int)

df_train["duration_days"]= df_train["duration_days"].fillna(365)

# Therapy phase: how long this patient has been in therapy (days since start).

df_train["therapy_phase"]= ((REF - df_train["TherapyStartDate"].fillna(REF - pd.Timedelta(days=365))).dt.days).clip(lower=1)

if "therapy_phase" not in feat_cols:feat_cols.append("therapy_phase")

print(f"\nTraining labels for features patients ({len(df_train):,}):")
print(f"y_natural:{df_train['y_natural'].sum():,} "f"({df_train['y_natural'].mean()*100:.1f}%)")
print(f"y_effectful: {df_train['y_effectful'].sum():,} "f"({df_train['y_effectful'].mean()*100:.1f}%)")
print(f"Overlap:{((df_train['y_natural']==1) & (df_train['y_effectful']==1)).sum():,} "f"(patients with both types in history)")
print(f"\nfeat_cols:{len(feat_cols)} features (including therapy_phase)")

Any-historical natural dropout:9,839 patients
Any-historical effectful dropout:10,663 patients

Training labels for features patients (41,115):
y_natural:46 (0.1%)
y_effectful: 47 (0.1%)
Overlap:0 (patients with both types in history)

feat_cols:49 features (including therapy_phase)


## LAYER 3 — TRAINING (Natural, Effectful, XGBoost TTD, Cox PH)

In [44]:
nat_det= NaturalDropoutDetector()
eff_det= EffectfulDropoutDetector()
ttd_model= TimeToDropoutModel()

#NATURAL DROPOUT DETECTOR

print("\nNatural Dropout Detector (LightGBM + asymmetric loss):")
nat_det.fit(df_train[feat_cols], df_train["y_natural"])

#EFFECTFUL DROPOUT DETECTOR

print("\nEffectful Dropout Detector (SMOTEBoost + real labels):")

eff_care_cols= [c for c in [
    "intervention_count_30d", "intervention_count_90d",
    "pending_count","pending_max_age",
    "days_since_last",
    "visit_count_30d","call_count_30d","sms_count_30d",
    "last_intervention_t1","last_intervention_t2","last_intervention_t3",
    "response_rate",
    "use_slope",
    "use_mean_7d",
    "cpap_macro_state",] if c in df_train.columns]

eff_det.fit(df_train[eff_care_cols], df_train["y_effectful"])

#TIME-TO-DROPOUT — XGBoost

print("\nTime-to-Dropout:")
xgb_mask= (df_train["y_natural"]== 1) | (df_train["y_effectful"] == 1)
print(f"XGBoost training: {xgb_mask.sum():,} dropout patients")

if xgb_mask.sum() >= 30:
    ttd_model.fit_xgb(df_train[xgb_mask][feat_cols],df_train[xgb_mask]["duration_days"],)
else:
    print(f"XGBoost:{xgb_mask.sum()} labels — skipping")

#TIME-TO-DROPOUT — Cox PH

if COX_OK and "duration_cox" in sd_cox.columns:
    sd_cox["event"]= ((sd_cox["true_natural"] == 1) | (sd_cox["true_effectful"] == 1)).astype(int)
    sd_cox_renamed= sd_cox.rename(columns={"duration_cox": "duration"})

    cox_feat_cols= [c for c in COX_FEATS if c in features.columns]
    survival_df= sd_cox_renamed[["AtHomePatientId", "duration", "event"]].merge(features[["AtHomePatientId"] + cox_feat_cols],on="AtHomePatientId", how="left",)

    print(f"Cox PH training: {len(survival_df):,} patients "f"({survival_df['event'].sum():,} events)")
    ttd_model.fit_cox(survival_df)
else:
    print(" Cox PH: skipped (COX_OK=False or duration_cox not in sd_cox)")


Natural Dropout Detector (LightGBM + asymmetric loss):

Effectful Dropout Detector (SMOTEBoost + real labels):
Effectful labels: 47/41,115 (0.1%)

Time-to-Dropout:
XGBoost training: 93 dropout patients
XGBoost TTD: MAE=800.9 days  n=74
Cox PH training: 26,341 patients (19,452 events)
 Cox PH: insufficient (0 rows, 0 events)


## LAYER 3 — COX PH BLOCK (age, gender, TherapyCode from Patients_all)

In [45]:
sd_cox= sd_cox.drop(columns=[c for c in ["age", "gender_enc", "age_std", "therapy_code_enc"]
             if c in sd_cox.columns])

therapy_code_map= {"SW": 0, "SL": 1, "SP": 2}
sd_cox["therapy_code_enc"]= sd_cox["TherapyCode"].map(therapy_code_map).fillna(0).astype(int)

# Survival event flag
sd_cox["event"]= ((sd_cox["true_natural"] == 1) | (sd_cox["true_effectful"] == 1)).astype(int)

#PATIENT DEMOGRAPHICS

patients_tmp= pd.read_csv("Patients all.csv",
    usecols=["AtHomePatientId", "BirthDate", "Gender"],)
patients_tmp["AtHomePatientId"] = patients_tmp["AtHomePatientId"].astype(int)
patients_tmp["age"]= ((REF - pd.to_datetime(patients_tmp["BirthDate"], errors="coerce")).dt.days / 365.25).clip(18, 95)
patients_tmp["gender_enc"]= (patients_tmp["Gender"] == "Male").astype(int)

sd_cox= sd_cox.merge(patients_tmp[["AtHomePatientId", "age", "gender_enc"]],on="AtHomePatientId", how="left",)
sd_cox["age"]= sd_cox["age"].fillna(sd_cox["age"].median())
sd_cox["gender_enc"]= sd_cox["gender_enc"].fillna(0).astype(int)

#AGE STANDARDISATION

from sklearn.preprocessing import StandardScaler as _SS
_age_scaler= _SS()
sd_cox["age_std"]= _age_scaler.fit_transform(sd_cox[["age"]]).ravel()
ttd_model._age_scaler= _age_scaler        
ttd_model._cox_feats_override= ["age_std","gender_enc","therapy_code_enc"]

#BUILD SURVIVAL_DF

dur_col= "duration_cox" if "duration_cox" in sd_cox.columns else "duration_days"

survival_df= (sd_cox[["age_std", "gender_enc", "therapy_code_enc", dur_col, "event"]].rename(columns={dur_col: "duration"}))

print(f"Cox PH survival_df: {len(survival_df):,} patients  "f"({survival_df['event'].sum():,} events  "f"| dur median={survival_df['duration'].median():.0f}d)")

ttd_model.fit_cox(survival_df)

Cox PH survival_df: 26,341 patients  (19,452 events  | dur median=631d)
 Cox PH: C-index=0.568  n=26,028 events=19,190


## LAYER 3 — RISK SCORES + PROXY BLEND + RESULTS

In [53]:
#NATURAL AND EFFECTFUL DROPOUT RISK
features["z_natural"]= nat_det.predict_risk(df_train[feat_cols])
features["z_effectful"]= eff_det.predict_effectful(df_train[eff_care_cols])

#TIME TO DROPOUT

if ttd_model.xgb_model is not None:
    _X_all= ttd_model.scaler.transform(df_train[feat_cols].fillna(0).values)
    _h_all= np.maximum(1.0, ttd_model.xgb_model.predict(_X_all))
else:
    _h_all= np.full(len(df_train), 365.0)

_hazard_all= np.full(len(df_train), 0.001)
if ttd_model.cox_model is not None and ttd_model._cox_cols:
    for _i, (_idx, _row) in enumerate(df_train.iterrows()):
        try:
            _therapy_days = float(df_train.loc[_idx, "therapy_phase"])
            _, _hazard_all[_i]= ttd_model.predict(_row, _therapy_days)
        except Exception:
            pass

features["h_days"]= _h_all
features["hazard_rate"]= _hazard_all

#COMBINED RISK SCORE

features["z_risk"]= (0.65 * features["z_natural"] +0.35 * features["z_effectful"]).clip(0, 1)

#RISK LEVEL — vectorized with np.select

features["risk_level"]= np.select(
    condlist=[features["z_risk"] >= 0.75,features["z_risk"] >= 0.55,features["z_risk"] >= 0.30,],
    choicelist=["critical", "high", "medium"],default="low",)

#DROPOUT MECHANISM — vectorized with np.select

features["dropout_mechanism"]= np.select(
    condlist=[features["z_risk"] < 0.30,features["z_effectful"] > 0.50,(features["z_effectful"] > 0.30) & (features["z_natural"] > 0.50),
    ],choicelist=["none", "effectful", "mixed"],default="natural",)

print(f"\nRisk score summary ({len(features):,} patients):")
print(f"z_natural: mean={features['z_natural'].mean():.3f} "f"std={features['z_natural'].std():.3f}")
print(f"z_effectful : mean={features['z_effectful'].mean():.3f}  "f"std={features['z_effectful'].std():.3f}")
print(f"  h_days: mean={features['h_days'].mean():.0f}d  "f"min={features['h_days'].min():.0f}d")
print(f"\nRisk level distribution:")
print(features["risk_level"].value_counts().to_string())
print(f"\nDropout mechanism distribution:")
print(features["dropout_mechanism"].value_counts().to_string())
#PROXY RISK SCORE
_macro_risk_map={0: 0.05, 1: 0.25, 2: 0.55, 3: 0.80}
_bio_risk_map={0: 0.0,  1: 0.2,  2: 0.5,  3: 0.8}

features["z_proxy"]= (features["cpap_macro_state"].map(_macro_risk_map).fillna(0.25) * 0.40
    + (-features["use_slope"].fillna(0) * 0.5).clip(0, 0.25) * 0.20
    + (features["low_use_streak"].fillna(0) / 5.0).clip(0, 1) * 0.15
    + (features["days_since_last"].fillna(999) / 90.0).clip(0, 1) * 0.10
    + (features["ESS_score"].fillna(0) / 24.0 * 0.5+ features["BDI_score"].fillna(0) / 63.0 * 0.5).clip(0, 1) * 0.10
    + (features["bio_state"].map(_bio_risk_map).fillna(0.0)* (features["bio_state"].fillna(-1) >= 0)) * 0.05).clip(0, 1)

n_labels= int(((df_train["y_natural"] == 1)|(df_train["y_effectful"] == 1)).sum())
ml_weight= min(0.8, n_labels / 500.0)
prx_weight= 1.0 - ml_weight

print(f" n_labels={n_labels:,} ml_weight={ml_weight:.2f} proxy_weight={prx_weight:.2f}")

features["z_risk"]= ( ml_weight*features["z_risk"] +prx_weight * features["z_proxy"]).clip(0, 1)

#RISK LEVEL + MECHANISM — np.select (vectorized)

features["risk_level"]= np.select(
    [features["z_risk"] >= 0.75, features["z_risk"] >= 0.55, features["z_risk"] >= 0.30],
    ["critical", "high", "medium"],default="low",)

features["dropout_mechanism"]= np.select(
    [features["z_risk"] < 0.30,features["z_effectful"] > 0.50,(features["z_effectful"] > 0.30) & (features["z_natural"] > 0.50),
    ],["none", "effectful", "mixed"],default="natural",)

print(f"\nFinal risk score after proxy blend:")
print(f" z_risk mean={features['z_risk'].mean():.3f} "f"z_proxy mean={features['z_proxy'].mean():.3f}")
print(f"\nRisk level:"); print(features["risk_level"].value_counts().to_string())
print(f"\nDropout mechanism:"); print(features["dropout_mechanism"].value_counts().to_string())
print(f"\nLayer 3: {len(features):,} patients")
print(f"ml_weight={ml_weight:.2f} proxy_weight={prx_weight:.2f}  "f"(n_labels={n_labels:,})")

#RISK LEVEL DISTRIBUTION

if "risk_level" in features.columns:
    print("\nRisk level distribution:")
    for lvl in ["low", "medium", "high", "critical"]:
        n= (features["risk_level"] == lvl).sum()
        print(f" {lvl:<10}: {n:>7,} ({n / len(features) * 100:.1f}%)")
else:
    print("\nrisk_level not yet computed — run the risk score block first.")

#DROPOUT MECHANISM

if "dropout_mechanism" in features.columns:
    print("\nDropout mechanism:")
    for m in ["none", "natural", "effectful", "mixed"]:
        n= (features["dropout_mechanism"] == m).sum()
        print(f"{m:<12}: {n:>7,}")

#Z_RISK BY CPAP MACRO-STATE

if "cpap_macro_state" in features.columns and "z_risk" in features.columns:
    print("\nz_risk by CPAP macro-state:")
    for k in range(4):
        sub= features[features["cpap_macro_state"] == k]["z_risk"]
        if len(sub)== 0:
            print(f"{CPAPSpecialist.MACRO_STATES[k]:<25}: (no patients)")
        else:
            print(f"{CPAPSpecialist.MACRO_STATES[k]:<25}: "
                  f"mean={sub.mean():.3f}  max={sub.max():.3f}")
else:
    print("\ncpap_macro_state or z_risk not yet computed — run Layer 2 first.")

#SAMPLE — 5 PoC PATIENTS

print("\nSample (5 PoC patients):")
poc_s= features[features["in_poc"]].head(5)

use_col= ("use_mean_7d" if "use_mean_7d" in poc_s.columns
           else "use_mean_x" if "use_mean_x" in poc_s.columns
           else "use_mean")
_sample_cols= ["AtHomePatientId", use_col, "cpap_macro_state","z_risk", "risk_level", "h_days", "dropout_mechanism",]
_sample_cols= [c for c in _sample_cols if c in poc_s.columns]

if _sample_cols:
    print(poc_s[_sample_cols].round(3).to_string(index=False))
else:
    print(" No displayable columns — run Layer 2 and Layer 3 first.")

_l3_cols= [
    "AtHomePatientId", "cpap_macro_state", "combined_state",
    "z_natural", "z_effectful", "z_proxy", "z_risk",
    "risk_level", "h_days", "hazard_rate", "dropout_mechanism","in_poc",]
_available= [c for c in _l3_cols if c in features.columns]
_missing= [c for c in _l3_cols if c not in features.columns]

if _missing:
    print(f" Missing columns (not yet computed): {_missing}")
    print(" Re-run Layer 2 and Layer 3 before exporting.")

layer3_results= features[_available].copy()

for col in _missing:
    layer3_results[col]= np.nan
layer3_results["cpap_macro_name"]= (layer3_results["cpap_macro_state"].map(CPAPSpecialist.MACRO_STATES).fillna("unknown"))

layer3_results.to_csv("layer3_results.csv", index=False)
print(f"Exported: layer3_results.csv| "f"{len(layer3_results):,} rows × {layer3_results.shape[1]} columns")
print(f"Exported: layer3_results.csv|"f"{len(layer3_results):,} rows × {layer3_results.shape[1]} columns")


Risk score summary (41,115 patients):
z_natural: mean=0.001 std=0.000
z_effectful : mean=0.271  std=0.181
  h_days: mean=983d  min=305d

Risk level distribution:
risk_level
low       40954
medium      161

Dropout mechanism distribution:
dropout_mechanism
none         40954
effectful      161
 n_labels=93 ml_weight=0.19 proxy_weight=0.81

Final risk score after proxy blend:
 z_risk mean=0.192 z_proxy mean=0.215

Risk level:
risk_level
low       32867
medium     8224
high         24

Dropout mechanism:
dropout_mechanism
none         32867
natural       5962
effectful     2286

Layer 3: 41,115 patients
ml_weight=0.19 proxy_weight=0.81  (n_labels=93)

Risk level distribution:
 low       :  32,867 (79.9%)
 medium    :   8,224 (20.0%)
 high      :      24 (0.1%)
 critical  :       0 (0.0%)

Dropout mechanism:
none        :  32,867
natural     :   5,962
effectful   :   2,286
mixed       :       0

z_risk by CPAP macro-state:
stable_adherent          : mean=0.096  max=0.187
moderate_user    

## LAYER 4 — INTERVENTION SIMULATOR

In [57]:
try:
    GAMMA
except NameError:
    GAMMA= 0.5
    INT_ENC= {"Visit": 0, "Call": 1, "Sms": 2, "SMS": 2, "Unknown": 3, "Not Defined": 3}
    INT_CAND= ["Visit", "Call", "Sms"]
    try:
        from xgboost import XGBRegressor
        XGB_OK= True
    except ImportError:
        XGB_OK= False
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    SEED= 42

class InterventionSimulator:

    def __init__(self, gamma=GAMMA):
        self.gamma= gamma
        self.risk_model= None
        self.horizon_model= None
        self.scaler= StandardScaler()
        self._cols= []
        self.failed_counts= {}   

    def build_triplets(self, interventions_df, usage_df,
                       pre_days=21, post_days=21, min_records=3):
        In_done= interventions_df[interventions_df["Status"] == "Done"].copy()
        In_done= In_done[["AtHomePatientId", "idate", "Category"]].copy()

        joined= In_done.merge(usage_df[["AtHomePatientId", "udate", "Use", "AHI"]],on="AtHomePatientId", how="inner",)
        joined["offset"]= (joined["udate"] - joined["idate"]).dt.days

        pre= (joined[(joined["offset"] >= -pre_days) & (joined["offset"] < 0)]
                .groupby(["AtHomePatientId", "idate"]).agg(use_pre=("Use","mean"), ahi_pre=("AHI","mean"),n_pre=("Use","count")).reset_index())
        post= (joined[(joined["offset"] > 2) & (joined["offset"] <= post_days)].groupby(["AtHomePatientId", "idate"])
                .agg(use_post=("Use","mean"), ahi_post=("AHI","mean"),n_post=("Use","count")).reset_index())

        triplets= (In_done.merge(pre,on=["AtHomePatientId", "idate"], how="inner").merge(post, on=["AtHomePatientId", "idate"], how="inner"))

        triplets= triplets[(triplets["n_pre"] >= min_records) &(triplets["n_post"] >= min_records)].copy()

        triplets["delta_use"]= triplets["use_post"] - triplets["use_pre"]
        triplets["delta_ahi"]= triplets["ahi_post"] - triplets["ahi_pre"]
        triplets["int_enc"]= triplets["Category"].map(INT_ENC).fillna(3).astype(int)
        return triplets

    def enrich_triplets(self, triplets, features_df):
        state_cols= [c for c in [
            "use_mean_7d", "use_slope", "ahi_mean_7d", "leaks_high_7d", "leaks95_mean_7d",
            "use_zscore", "ahi_zscore", "low_use_streak", "instability_idx",
            "intervention_count_30d", "days_since_last", "response_rate",
            "last_intervention_t1", "last_intervention_t2", "last_intervention_t3",
            "ESS_score", "ESS_missing", "BDI_score", "BDI_missing",
            "cpap_macro_state", "z_risk", "rho",
        ] if c in features_df.columns]

        for old, new in [("use_mean_x","use_mean_7d"),("ahi_mean_x","ahi_mean_7d"),
                         ("leaks95_mean_x","leaks95_mean_7d"),
                         ("leaks95_mean_x","leaks_high_7d")]:
            if new not in state_cols and old in features_df.columns:
                state_cols.append(old)

        enriched= triplets.merge(
            features_df[["AtHomePatientId"] + state_cols],
            on="AtHomePatientId", how="left",)
        return enriched, state_cols

    def fit(self, triplets):
        feature_cols= [c for c in triplets.columns if c not in [
            "AtHomePatientId",
            "idate", "udate", "Category",
            "delta_use", "delta_ahi",
            "use_pre", "ahi_pre", "use_post", "ahi_post",
            "n_pre", "n_post",]]
        self._cols= feature_cols

        X= triplets[feature_cols].fillna(0)
        Xs= self.scaler.fit_transform(X)

        use_std= triplets["delta_use"].std()
        y_risk= (-triplets["delta_use"] / (use_std + 1e-6)).values
        y_horizon= ( triplets["delta_use"] * 7).values

        X_tr, X_va, yr_tr, yr_va, yh_tr, yh_va = train_test_split(
            Xs, y_risk, y_horizon, test_size=0.2, random_state=SEED)

        model_kw= dict(n_estimators=300, learning_rate=0.05, max_depth=5,
                        subsample=0.8, colsample_bytree=0.8,
                        reg_alpha=0.1, reg_lambda=1.0,
                        random_state=SEED, n_jobs=-1)

        if XGB_OK:
            try:
                self.risk_model= XGBRegressor(early_stopping_rounds=30, **model_kw)
                self.horizon_model= XGBRegressor(early_stopping_rounds=30, **model_kw)
                self.risk_model.fit(X_tr, yr_tr, eval_set=[(X_va, yr_va)], verbose=False)
                self.horizon_model.fit(X_tr, yh_tr, eval_set=[(X_va, yh_va)], verbose=False)
            except TypeError:
                self.risk_model= XGBRegressor(**model_kw)
                self.horizon_model= XGBRegressor(**model_kw)
                self.risk_model.fit(X_tr, yr_tr, eval_set=[(X_va, yr_va)],early_stopping_rounds=30, verbose=False)
                self.horizon_model.fit(X_tr, yh_tr, eval_set=[(X_va, yh_va)],early_stopping_rounds=30, verbose=False)
        else:
            gbr_kw= {k: v for k, v in model_kw.items()
                      if k in ["n_estimators","learning_rate","max_depth","random_state"]}
            self.risk_model= GradientBoostingRegressor(**gbr_kw)
            self.horizon_model= GradientBoostingRegressor(**gbr_kw)
            self.risk_model.fit(X_tr, yr_tr)
            self.horizon_model.fit(X_tr, yh_tr)

        yr_pred= self.risk_model.predict(X_va)
        yh_pred= self.horizon_model.predict(X_va)
        print(f" Risk model MAE:{np.mean(np.abs(yr_pred - yr_va)):.4f}")
        print(f"Horizon model MAE: {np.mean(np.abs(yh_pred - yh_va)):.2f} days")

    def predict_delta(self, state_dict, intervention):
        state_dict["int_enc"]= float(INT_ENC.get(intervention, 3))
        x= np.nan_to_num(np.array([state_dict.get(c, 0.0) for c in self._cols], dtype=float),nan=0.0,).reshape(1, -1)
        xs= self.scaler.transform(x)
        return (float(self.risk_model.predict(xs)[0]),float(self.horizon_model.predict(xs)[0]))

    def select_intervention(self, patient_id, state_dict, z_risk, h_days,candidates=None):
        if candidates is None:
            candidates= INT_CAND

        if z_risk < 0.10:
            return None, "routine_monitoring", pd.DataFrame()

        effective_gamma= self.gamma * (365.0 / max(float(h_days), 30.0))
        failed= self.failed_counts.get(int(patient_id), {})
        sims= []

        for intv in candidates:
            delta_risk, delta_horizon= self.predict_delta(dict(state_dict), intv)
            score= delta_horizon - effective_gamma * delta_risk
            fail_count= failed.get(intv, 0)
            feasible= (delta_risk < 0) and (delta_horizon > 0) and (fail_count < 2)
            sims.append({"intervention": intv, "delta_risk": round(delta_risk, 4),"delta_horizon": round(delta_horizon, 2), "score": round(score, 4),
                         "fail_count": fail_count, "feasible": feasible})

        sims_df= pd.DataFrame(sims)
        feasible= sims_df[sims_df["feasible"]]

        if len(feasible)== 0:
            return None,"physician_escalation", sims_df

        best = feasible.loc[feasible["score"].idxmax()]
        return best["intervention"], "proceed", sims_df

    def record_outcome(self, patient_id, intervention, delta_use):
        pid = int(patient_id)
        if pid not in self.failed_counts:
            self.failed_counts[pid]= {}
        if delta_use < -0.5:
            self.failed_counts[pid][intervention]= \
                self.failed_counts[pid].get(intervention, 0) + 1
        else:
            self.failed_counts[pid][intervention]= 0

## LAYER 4 — BUILD TRIPLETS + TRAIN

In [58]:
#INTERVENTION DATA

In_raw= pd.read_csv("Intervention3.csv", engine="python", on_bad_lines="skip")
In_raw= In_raw.rename(columns={"s": "Status"})
In_raw["idate"] = pd.to_datetime(In_raw["ReferenceDate"].astype(str).str.split(".").str[0],errors="coerce",)

n_before= len(In_raw)
In_raw= In_raw[
    In_raw["idate"].notna() &
    (In_raw["idate"] >= "2020-01-01") &
    (In_raw["idate"] <= "2030-12-31")].copy()
print(f"Interventions loaded: {n_before:,} total → {len(In_raw):,} in 2024-12 to 2025-04")

In_raw["AtHomePatientId"]= In_raw["AtHomePatientId"].astype(int)

#INTERVENTION CATEGORY MAPPING
try:
    int_def= pd.read_csv("Interventiondefinition.csv")
    int_def["Category"]= int_def["Category"].str.strip().str.title()
    In_raw = pd.merge(In_raw, int_def, on="JobTypeCode", how="left")
    In_raw["Category"]= In_raw["Category"].fillna("Unknown")
    print(f" Category distribution: {In_raw['Category'].value_counts().to_dict()}")
except FileNotFoundError:
    print(" Interventiondefinition.csv not found — using 'Unknown' for all categories")
    In_raw["Category"]= "Unknown"

#USAGE DATA

usage_raw= pd.read_csv("Usage3.csv",usecols=["AtHomePatientId", "ReferenceDate", "Use", "AHI"],)
usage_raw= usage_raw[usage_raw["AtHomePatientId"].notna()].copy()
usage_raw["AtHomePatientId"] = usage_raw["AtHomePatientId"].astype(int)

usage_raw["udate"] = pd.to_datetime(usage_raw["ReferenceDate"].fillna(0).astype(int).astype(str),format="%Y%m%d", errors="coerce",)
usage_raw= usage_raw[
    usage_raw["udate"].notna() &
    (usage_raw["udate"]>= "2020-01-01") &
    (usage_raw["udate"]<= "2030-12-31")][["AtHomePatientId", "udate", "Use", "AHI"]]
print(f"Usage records: {len(usage_raw):,}")

simulator= InterventionSimulator(gamma=GAMMA)
print("\nBuilding (state, intervention, outcome) triplets...")
triplets= simulator.build_triplets(In_raw, usage_raw)
print(f" Triplets: {len(triplets):,} | Patients: {triplets['AtHomePatientId'].nunique():,}")
print(" By intervention type:")
for cat, n in triplets["Category"].value_counts().items():
    mu= triplets[triplets["Category"]== cat]["delta_use"].mean()
    print(f"{cat:<12}: {n:>6,} delta_use_mean={mu:+.3f}h")

#ENRICH + TRAIN

print("\nEnriching triplets with patient state features...")
triplets_enriched, state_cols = simulator.enrich_triplets(triplets, features)
triplets_enriched = triplets_enriched.dropna(subset=["delta_use"])
print(f"  Enriched triplets: {len(triplets_enriched):,} "f"features: {len(state_cols) + 3}")

if len(triplets_enriched) >= 50:
    print("\nTraining Layer 4 XGBoost models...")
    simulator.fit(triplets_enriched)
else:
    print(f"\nOnly {len(triplets_enriched)} enriched triplets — "f"need ≥ 50 to train.  Check intervention/usage date overlap.")

triplets_enriched.to_csv("layer4_triplets.csv", index=False)
print(f"Saved: layer4_triplets.csv | {len(triplets_enriched):,} rows ""(re-run Step 5 / Care Features after this cell to pick up real response_rate)")

Interventions loaded: 54,162 total → 49,978 in 2024-12 to 2025-04
 Category distribution: {'Visit': 34575, 'Call': 10762, 'Sms': 4546, 'Not Defined': 93, 'Unknown': 2}
Usage records: 3,463,318

Building (state, intervention, outcome) triplets...
 Triplets: 30,182 | Patients: 16,038
 By intervention type:
Visit       : 19,721 delta_use_mean=+0.118h
Call        :  7,675 delta_use_mean=+0.148h
Sms         :  2,742 delta_use_mean=+0.448h
Not Defined :     42 delta_use_mean=+0.535h
Unknown     :      2 delta_use_mean=+0.254h

Enriching triplets with patient state features...
  Enriched triplets: 30,182 features: 25

Training Layer 4 XGBoost models...
 Risk model MAE:0.5008
Horizon model MAE: 4.68 days
Saved: layer4_triplets.csv | 30,182 rows (re-run Step 5 / Care Features after this cell to pick up real response_rate)


## GOVERNANCE — GovernanceMonitor + run

In [59]:
try:
    import shap
    SHAP_OK= True
except ImportError:
    SHAP_OK= False

try:
    import lightgbm as lgb
    LGB_OK= True
except ImportError:
    LGB_OK= False

from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import datetime

@dataclass
class GovernanceReport:
    timestamp:str
    n_patients: int
    override_rate: float
    calibration_ece:float
    calibration_ok:bool
    intervention_benefit:float
    benefit_by_type:Dict
    cluster_K_history:List
    cluster_status:str
    shap_drift:float
    shap_top_changed:bool
    horizon_mae:float
    actions_required:List[str]= field(default_factory=list)
    retrain_priority:List[str]= field(default_factory=list)

class GovernanceMonitor:

    THRESHOLDS = {
        "override_rate":0.20,
        "calibration_ece":0.10,
        "horizon_mae_days":30,
        "cpap_silhouette":0.40,
        "effectful_recall":0.60,
        "intervention_benefit":0.0,
        "patient_burden":3.0,}

    def __init__(self):
        self.K_history: Dict[str, List]= {"cpap": []}
        self.shap_baseline:Optional[np.ndarray]= None
        self.shap_feat_names: List[str]= []
        self.override_log:List[dict]= []
        self.run_history:List[GovernanceReport]= []
    def monitor_calibration(self, y_true: np.ndarray,y_pred_proba: np.ndarray,n_bins: int = 10) -> float:
        if y_true.sum() < 5:
            return float("nan")
        try:
            frac_pos, mean_pred= calibration_curve(
                y_true, y_pred_proba, n_bins=n_bins, strategy="quantile")
            return float(np.mean(np.abs(frac_pos - mean_pred)))
        except Exception:
            return float("nan")

    def set_shap_baseline(self, model, X_sample: np.ndarray,feature_names: List[str]):
        if not SHAP_OK:
            return
        try:
            explainer= shap.TreeExplainer(model)
            sv= explainer.shap_values(X_sample)
            sv= np.abs(sv[1] if isinstance(sv, list) else sv).mean(axis=0)
            self.shap_baseline= sv/ (sv.sum() + 1e-10)
            self.shap_feat_names= feature_names
        except Exception as e:
            print(f" SHAP baseline failed: {e}")

    def compute_shap_drift(self, model, X_new: np.ndarray) -> tuple:
        if not SHAP_OK or self.shap_baseline is None:
            return 0.0, False
        try:
            explainer= shap.TreeExplainer(model)
            sv= explainer.shap_values(X_new)
            sv= np.abs(sv[1] if isinstance(sv, list) else sv).mean(axis=0)
            sv_norm= sv / (sv.sum() + 1e-10)
            drift= float(np.mean(np.abs(sv_norm - self.shap_baseline) /(self.shap_baseline + 1e-6)))
            top_changed = len(set(np.argsort(sv_norm)[-5:]) -set(np.argsort(self.shap_baseline)[-5:])) >= 3
            return drift, top_changed
        except Exception:
            return 0.0, False

    #CLUSTER STABILITY

    def check_cluster_stability(self, specialist: str, K_new: int) -> str:
        self.K_history[specialist].append(K_new)
        recent= self.K_history[specialist][-3:]
        if len(recent)== 3 and len(set(recent))== 1:
            return "stable_convert_to_classification"
        if len(recent)>= 2 and recent[-1]!= recent[-2]:
            return "unstable_rebuild_matrix"
        return "stable"

    #INTERVENTION BENEFIT

    def check_intervention_benefit(self, triplets_df: pd.DataFrame) -> dict:
        benefit= {}
        for intv, grp in triplets_df.groupby("Category"):
            benefit[intv]= {"mean_delta_use": round(float(grp["delta_use"].mean()), 3),"n":len(grp),"positive_rate":  round(float((grp["delta_use"] > 0.5).mean()), 3),}
        return {"by_type": benefit,"overall": round(float(triplets_df["delta_use"].mean()), 3)}

    #HORIZON MAE

    def check_horizon_mae(self, ttd_model, X_test: pd.DataFrame,y_days_true: pd.Series) -> float:
        if ttd_model.xgb_model is None or len(X_test) < 5:
            return float("nan")
        try:
            y_pred= ttd_model.xgb_model.predict(
                ttd_model.scaler.transform(X_test.fillna(0).values))
            return float(np.mean(np.abs(y_pred - y_days_true.values)))
        except Exception:
            return float("nan")

    #REPTILE META-LEARNING ADAPTATION

    def reptile_adapt(self, nat_detector, X_new: pd.DataFrame, y_new: pd.Series, n_inner_steps: int = 5):
        if not LGB_OK or nat_detector.model is None:
            print(" Reptile: LightGBM model not available")
            return nat_detector

        print(f" Reptile adaptation: {len(X_new):,} new samples, "
              f"{int(y_new.sum())} new dropout labels")

        X_s= nat_detector.scaler.transform(X_new[nat_detector._cols].fillna(0).values)
        dtask= lgb.Dataset(X_s, label=y_new.values)

        params = {
            "objective":nat_detector.asymmetric_loss,
            "num_leaves":31,
            "learning_rate":0.005,
            "min_child_samples": 10,
            "seed":SEED,
            "verbosity":-1,}
        nat_detector.model= lgb.train(params, dtask,num_boost_round= n_inner_steps * 10,init_model = nat_detector.model,)
        print(f"Reptile: model updated with {n_inner_steps * 10} additional trees")
        return nat_detector

    def maml_inner_update(self, params: dict, gradients: dict,lr: float = 0.01) -> dict:
        """Conceptual MAML inner loop for tree models."""
        return {k: v - lr * gradients.get(k, 0) for k, v in params.items()}

    def compute_retrain_priority(self, metrics: dict) -> List[str]:
        scores = {"natural_dropout":   (0.0 if np.isnan(metrics.get("ece", 0.5))
                                  else max(0, metrics.get("ece", 0.5) -self.THRESHOLDS["calibration_ece"])),
            "effectful_dropout":max(0, 0.6 - metrics.get("effectful_recall", 0.6)),
            "cpap_clustering":max(0, self.THRESHOLDS["cpap_silhouette"] -metrics.get("silhouette", 0.35)),
            "intervention_sim":max(0, -metrics.get("intervention_benefit", 0.0)),
            "ttd_model":max(0, (metrics.get("horizon_mae", 15) -self.THRESHOLDS["horizon_mae_days"]) / 30),}
        return sorted(scores, key=scores.get, reverse=True)

    def log_override(self, patient_id: int, system_rec: str,actual_decision: str, source: str = "technician"):
        self.override_log.append({
            "timestamp": datetime.now().isoformat(),
            "patient_id":patient_id,
            "system_rec":system_rec,
            "actual":actual_decision,
            "overridden":system_rec != actual_decision,
            "source":source,})

    def override_rate(self, last_n: int = 100) -> float:
        log = self.override_log[-last_n:]
        if not log:
            return 0.0
        return float(sum(1 for r in log if r["overridden"]) / len(log))

    def run(self, features_df: pd.DataFrame, df_train: pd.DataFrame,nat_detector, eff_detector, cpap_specialist,ttd_model, triplets_df: pd.DataFrame) -> GovernanceReport:
        print("\nRunning governance checks...")
        actions = []
        metrics = {}

        # 1. CALIBRATION
        print(" [1/6] Calibration (ECE)...")
        y_true= df_train["y_natural"].values
        feat_c= [c for c in nat_detector._cols if c in df_train.columns]
        if feat_c and nat_detector.model is not None:
            z_pred= nat_detector.predict_risk(df_train[feat_c])
            ece= self.monitor_calibration(y_true, z_pred)
            metrics["ece"]= ece
            cal_ok= np.isnan(ece) or ece <= self.THRESHOLDS["calibration_ece"]
            status_ece= f"{ece:.4f}" if not np.isnan(ece) else "n/a"
            print(f"ECE = {status_ece}  {'✓' if cal_ok else '⚠ recalibrate'}")
            if not cal_ok:
                actions.append(f"Recalibrate natural dropout model (ECE={ece:.3f})")
        else:
            ece= float("nan"); cal_ok = True
            print("Skipped (model not fitted)")

        # 2. INTERVENTION BENEFIT
        print("[2/6] Intervention benefit...")
        benefit = self.check_intervention_benefit(triplets_df)
        metrics["intervention_benefit"]= benefit["overall"]
        print(f"Overall mean Δuse= {benefit['overall']:+.3f} h/night")
        for intv, b in benefit["by_type"].items():
            flag= "⚠" if b["mean_delta_use"] < 0 else "✓"
            print(f"{intv:<12} {b['mean_delta_use']:+.3f}h  "
                  f"pos_rate={b['positive_rate']:.1%}  n={b['n']}  {flag}")
        if benefit["overall"] < self.THRESHOLDS["intervention_benefit"]:
            actions.append("Intervention benefit negative — review selection logic")

        # 3. CLUSTER STABILITY
        print("[3/6] Cluster stability...")
        if cpap_specialist.cluster_history:
            K_now= cpap_specialist.cluster_history[-1]
            status= self.check_cluster_stability("cpap", K_now)
            metrics["cluster_status"] = status
            print(f"K_history = {self.K_history['cpap']}  → {status}")
            if status== "unstable_rebuild_matrix":
                actions.append("Clustering unstable — rebuild decision matrix")
            elif status== "stable_convert_to_classification":
                print(" K stable — ready to switch to supervised classification")
        else:
            status ="no_history"
            print("No cluster history yet")

        # 4. SHAP DRIFT
        print("[4/6] SHAP feature drift...")
        if SHAP_OK and nat_detector.model is not None:
            feat_cols= [c for c in nat_detector._cols if c in features_df.columns]
            if feat_cols and self.shap_baseline is None:
                X_s= nat_detector.scaler.transform(
                    features_df[feat_cols].fillna(0).values[:500])
                self.set_shap_baseline(nat_detector.model, X_s, feat_cols)
                drift, top_changed = 0.0, False
                print("Baseline set (first run)")
            elif feat_cols:
                X_s= nat_detector.scaler.transform(
                    features_df[feat_cols].fillna(0).values[:500])
                drift, top_changed = self.compute_shap_drift(nat_detector.model, X_s)
                metrics["shap_drift"] = drift
                flag = "drift detected" if drift > 0.20 else "✓"
                print(f"drift={drift:.4f}  top_changed={top_changed}  {flag}")
                if drift > 0.20:
                    actions.append(f"SHAP drift ({drift:.3f}) — schedule retrain")
            else:
                drift, top_changed= 0.0, False
                print("Skipped")
        else:
            drift, top_changed= 0.0, False
            print("SHAP not available — skipped")

        # 5. HORIZON MAE
        print("[5/6] Time-to-dropout MAE...")
        dropout_mask= df_train["y_natural"] == 1
        if dropout_mask.sum() >= 10 and ttd_model.xgb_model is not None:
            feat_ttd= [c for c in ttd_model._cols if c in df_train.columns]
            mae= self.check_horizon_mae(ttd_model,df_train[dropout_mask][feat_ttd],df_train[dropout_mask]["duration_days"],)
            metrics["horizon_mae"]= mae
            if np.isnan(mae):
                print("n/a")
            elif mae > self.THRESHOLDS["horizon_mae_days"]:
                print(f"⚠ MAE={mae:.0f}d > {self.THRESHOLDS['horizon_mae_days']}d")
                actions.append(f"TTD model MAE={mae:.0f}d — retrain with more labels")
            else:
                print(f"✓ MAE={mae:.0f}d")
        else:
            mae= float("nan")
            print(f"Skipped ({dropout_mask.sum()} dropout labels)")

        # 6. OVERRIDE RATE
        print("[6/6] Override rate...")
        ovr= self.override_rate()
        metrics["override_rate"]= ovr
        print(f"Override rate= {ovr:.1%} "f"(logged {len(self.override_log)} decisions)")
        if ovr > self.THRESHOLDS["override_rate"]:
            actions.append(f"Override rate {ovr:.1%} > 20% — model trust low")

        priority= self.compute_retrain_priority(metrics)

        report= GovernanceReport(
            timestamp= datetime.now().isoformat(),
            n_patients= len(features_df),
            override_rate= ovr,
            calibration_ece= ece,
            calibration_ok= cal_ok,
            intervention_benefit= benefit["overall"],
            benefit_by_type= benefit["by_type"],
            cluster_K_history= list(self.K_history["cpap"]),
            cluster_status= status,
            shap_drift= drift,
            shap_top_changed= top_changed,
            horizon_mae= mae,
            actions_required= actions,
            retrain_priority= priority,)
        self.run_history.append(report)
        return report

#RUN GOVERNANCE

gov= GovernanceMonitor()

report= gov.run(
    features_df= features,
    df_train= df_train,
    nat_detector= nat_det,
    eff_detector= eff_det,
    cpap_specialist= cpap_sp,
    ttd_model= ttd_model,
    triplets_df= triplets,)

print(f'\n{"="*55}')
print(f'GOVERNANCE REPORT — {report.timestamp[:10]}')
print(f'{"="*55}')
print(f'Patients monitored: {report.n_patients:,}')
print(f'Calibration ECE: {report.calibration_ece:.4f} 'f'{"OK" if report.calibration_ok else "NEEDS RECALIBRATION"}')
print(f'Intervention benefit: {report.intervention_benefit:+.3f} h/night')
print(f'SHAP drift: {report.shap_drift:.4f}')
if not np.isnan(report.horizon_mae):
    print(f'Horizon MAE: {report.horizon_mae:.1f} days')
else:
    print(f'Horizon MAE: n/a')
print(f'Override rate: {report.override_rate:.1%}')

print(f'\nRetrain priority order (TAML):')
for i, sp in enumerate(report.retrain_priority, 1):
    print(f'{i}. {sp}')

if report.actions_required:
    print(f'\nActions required ({len(report.actions_required)}):')
    for a in report.actions_required:
        print(f' ⚠ {a}')
else:
    print(f'\n✓ All systems nominal — no actions required')

poc_mask= df_train["AtHomePatientId"].isin(
    features[features["in_poc"]]["AtHomePatientId"].tolist())
if poc_mask.sum() >= 20:
    print(f'\nReptile adaptation on PoC cohort ({poc_mask.sum()} patients)...')
    feat_c= [c for c in nat_det._cols if c in df_train.columns]
    nat_det = gov.reptile_adapt(
        nat_det,
        df_train[poc_mask][feat_c],
        df_train[poc_mask]["y_natural"],
        n_inner_steps=5,)
else:
    print(f'\nReptile: insufficient PoC labels ({poc_mask.sum()}) — skipped')


Running governance checks...
 [1/6] Calibration (ECE)...
ECE = 0.0062  ✓
[2/6] Intervention benefit...
Overall mean Δuse= +0.156 h/night
Call         +0.148h  pos_rate=28.1%  n=7675  ✓
Not Defined  +0.535h  pos_rate=45.2%  n=42  ✓
Sms          +0.448h  pos_rate=42.2%  n=2742  ✓
Unknown      +0.254h  pos_rate=50.0%  n=2  ✓
Visit        +0.118h  pos_rate=25.0%  n=19721  ✓
[3/6] Cluster stability...
K_history = [71]  → stable
[4/6] SHAP feature drift...
SHAP not available — skipped
[5/6] Time-to-dropout MAE...
⚠ MAE=380d > 30d
[6/6] Override rate...
Override rate= 0.0% (logged 0 decisions)

GOVERNANCE REPORT — 2026-07-16
Patients monitored: 41,115
Calibration ECE: 0.0062 OK
Intervention benefit: +0.156 h/night
SHAP drift: 0.0000
Horizon MAE: 380.0 days
Override rate: 0.0%

Retrain priority order (TAML):
1. ttd_model
2. cpap_clustering
3. natural_dropout
4. effectful_dropout
5. intervention_sim

Actions required (1):
 ⚠ TTD model MAE=380d — retrain with more labels

Reptile adaptation on 

In [ ]:
#Intervention selection-Video

In [64]:
#SURVEY CLINICAL MAPPING

SURVEY_CLINICAL_MAP= {
    "ESS":{
        "detects":"Excessive daytime sleepiness — primary CPAP efficacy marker",
        "trigger_if": "ESS_missing=1 OR (ESS_score≥8 AND ESS_freshness<0.5)",
        "urgency": "high",},
    "ISI": {
        "detects": "Insomnia severity — residual sleep initiation/maintenance issues",
        "trigger_if": "ISI_missing=1  OR  ISI_score≥8",
        "urgency":"high",},
    "BDI": {
        "detects": "Depression — strong predictor of CPAP abandonment",
        "trigger_if": "BDI_missing=1  OR  BDI_score≥11  OR  dropout_mechanism contains 'natural'",
        "urgency": "high",},
    "PSQI": {
        "detects":   "Overall sleep quality — masks poor sleep despite CPAP use",
        "trigger_if": "PSQI_missing=1  OR  PSQI_score≥6",
        "urgency":   "medium",},
    "FSS": {
        "detects": "Fatigue severity — chronic fatigue independent of sleepiness",
        "trigger_if": "FSS_missing=1 AND  (ESS_score<8 but patient still complains)",
        "urgency": "medium",},
    "SF36": {
        "detects": "General quality of life — overall health perception",
        "trigger_if": "SF36_missing=1  AND  high z_risk",
        "urgency": "low",},}

#BIOMARKER CLINICAL MAP
BIOMARKER_MAP = {
    "SpO2 monitoring (Masimo/WW)": {"trigger": "spo2_7d < 90  OR  AHI residual > 10  OR  critical_alert",
        "reason": "Residual nocturnal desaturations despite CPAP — may need pressure adjustment",},
    "HRV monitoring (Hexoskin)": {"trigger": "hrv_rmssd_7d < 15  OR  hrv_lf_hf_ratio_7d > 3",
        "reason":  "Autonomic stress pattern — CPAP not fully resolving cardiovascular load",},
    "Sleep staging (SomnoArt)": {"trigger": "sleep_efficiency_7d < 0.65  OR  n3_duration_7d < 30",
        "reason":  "Fragmented sleep architecture — may indicate residual apnea or insomnia",},
    "Blood pressure (BPM)": {"trigger": "systolic_bp_7d > 140  OR  afib_detected = 1",
        "reason":  "Uncontrolled hypertension or arrhythmia — needs physician evaluation",},}

#LAYER 5 RECOMMENDER CLASS
class InterventionDataRecommender:
    """
    For each patient recommends:
    1. Intervention channel (Visit/Call/SMS/Video)
    2. Surveys to collect
    3. Biomarker monitoring to request
    """

    @staticmethod
    def _f(row, key, default=0.0):
        """NaN-safe float fetch — returns default for NaN/None/missing."""
        v = row.get(key)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return float(default)
        try:
            return float(v)
        except (ValueError, TypeError):
            return float(default)

    @staticmethod
    def _i(row, key, default=0):
        """NaN-safe int fetch."""
        v = row.get(key)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return int(default)
        try:
            return int(float(v))
        except (ValueError, TypeError):
            return int(default)

    @staticmethod
    def _s(row, key, default=""):
        """NaN-safe string fetch."""
        v = row.get(key)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return str(default)
        return str(v)

    def recommend_intervention(self, row):
        z_risk= self._f(row, "z_risk", 0)
        z_eff= self._f(row, "z_effectful",0)
        z_nat= self._f(row, "z_natural", 0)
        mechanism= self._s(row, "dropout_mechanism", "none")
        routing= self._s(row, "routing","proceed")
        resp_rate= self._f(row, "response_rate", 0.5)
        macro= self._i(row, "cpap_macro_state", 1)

        if routing== "technician_alert":
            return "Visit", "No CPAP transmission in 7d — technician must check device & mask"
        if routing== "sync_alert":
            return "Call", "No care contact in 90d — re-establish contact before escalating"

        if z_risk < 0.30:
            return "SMS", "Low risk — light touch engagement"

        if mechanism== "effectful":
            if z_risk >= 0.55:
                return "Visit", "High effectful risk — face-to-face needed to address care breakdown"
            return "Call", "Moderate effectful risk — call to check care satisfaction"

        if mechanism== "natural":
            if resp_rate < 0.25:
                return "Video", "Low response to previous contacts — video as low-burden alternative"
            if z_nat >= 0.55:
                return "Call", "Patient disengaging — call to understand barriers"
            return "SMS", "Natural risk — motivational SMS to reinforce therapy value"

        if mechanism== "mixed":
            if z_risk >= 0.55:
                return "Visit", "Mixed high risk — in-person needed"
            return "Call", "Mixed moderate risk — call to triage driver"

        if macro== 3:
            return "Visit", "Critical non-adherent — direct visit to prevent imminent dropout"
        if macro== 2:
            return "Call", "Declining user — call to identify cause"
        return "SMS", "Moderate/stable — lightweight engagement"

    def recommend_surveys(self, row):
        surveys= []
        reasons= {}
        mechanism= self._s(row, "dropout_mechanism", "none")
        z_nat= self._f(row, "z_natural",0)
        macro= self._i(row, "cpap_macro_state",1)

        ess_miss= self._i(row, "ESS_missing",1)
        ess_fresh= self._f(row, "ESS_freshness",0)
        ess_score= self._f(row, "ESS_score",0)
        bdi_miss= self._i(row, "BDI_missing",1)
        bdi_score= self._f(row, "BDI_score",0)
        isi_miss= self._i(row, "ISI_missing",1)
        isi_score= self._f(row, "ISI_score",0)
        psqi_miss= self._i(row, "PSQI_missing",1)
        fss_miss= self._i(row, "FSS_missing",1)
        sf36_miss= self._i(row, "SF36_missing",1)
        ahi=self._f(row, "ahi_mean_7d",5)
        z_risk= self._f(row, "z_risk",0)

        if ess_miss== 1:
            surveys.append("ESS")
            reasons["ESS"] = "Missing — primary CPAP efficacy marker"
        elif ess_fresh < 0.4:
            surveys.append("ESS")
            reasons["ESS"]= "Stale (>6 months) — recheck daytime sleepiness"

        if bdi_miss== 1 and (z_nat > 0.20 or mechanism == "natural"):
            surveys.append("BDI")
            reasons["BDI"] = "Natural dropout risk — depression screen critical"
        elif bdi_score >= 11:
            surveys.append("BDI")
            reasons["BDI"]= f"Previous BDI={bdi_score:.0f} ≥11 — monitor depression"

        if isi_miss== 1 and macro >= 2:
            surveys.append("ISI")
            reasons["ISI"]= "Declining/critical — insomnia may drive usage drop"
        elif isi_score >= 8:
            surveys.append("ISI")
            reasons["ISI"]= f"ISI={isi_score:.0f} ≥8 — subthreshold insomnia confirmed"

        if psqi_miss== 1 and ahi < 5 and macro >= 1:
            surveys.append("PSQI")
            reasons["PSQI"]= "AHI controlled but unstable — PSQI reveals sleep quality issues"

        if fss_miss== 1 and ess_score < 8 and macro >= 2:
            surveys.append("FSS")
            reasons["FSS"]= "ESS controlled but declining — fatigue (not sleepiness) may be driver"

        if not surveys and z_risk >= 0.55:
            surveys.append("SF36")
            reasons["SF36"]= "High risk, all scales present — QoL baseline for escalation"

        return surveys[:3],reasons

    def recommend_biomarker(self, row):
        bio_abs= self._i(row, "bio_absent", 1)
        requests= []
        reasons= {}

        if bio_abs== 0:
            spo2= self._f(row, "spo2_7d",95.0)
            hrv= self._f(row, "hrv_rmssd_7d",30.0)
            ahi=self._f(row, "ahi_mean_7d",5.0)
            sleep_eff= self._f(row, "sleep_efficiency_7d",0.85)
            sbp= self._f(row, "systolic_bp_7d",120.0)
            afib= self._i(row, "afib_detected",0)

            if spo2 < 92 or ahi > 10:
                requests.append("SpO2 monitoring (Masimo/WW)")
                reasons["SpO2 monitoring (Masimo/WW)"]= \
                    f"SpO2={spo2:.0f}% / residual AHI={ahi:.1f} — nocturnal desaturation likely"

            if hrv < 15:
                requests.append("HRV monitoring (Hexoskin)")
                reasons["HRV monitoring (Hexoskin)"]= \
                    f"HRV RMSSD={hrv:.0f}ms very low — cardiovascular stress unresolved"

            if sleep_eff < 0.65:
                requests.append("Sleep staging (SomnoArt)")
                reasons["Sleep staging (SomnoArt)"]= \
                    f"Sleep efficiency={sleep_eff:.0%} — fragmented sleep despite CPAP"

            if afib== 1 or sbp > 150:
                requests.append("Blood pressure (BPM)")
                reasons["Blood pressure (BPM)"]= \
                    "AFib or hypertension detected — cardiac risk assessment needed"

        else:
            ahi= self._f(row, "ahi_mean_7d", 5.0)
            z_risk= self._f(row, "z_risk", 0.0)
            z_natural= self._f(row, "z_natural", 0.0)
            macro= self._i(row, "cpap_macro_state", 1)
            ess= self._f(row, "ESS_score", 0.0)
            bdi= self._f(row, "BDI_score", 0.0)
            isi= self._f(row, "ISI_score", 0.0)

            if ahi > 15:
                requests.append("SpO2 monitoring (Masimo/WW)")
                reasons["SpO2 monitoring (Masimo/WW)"]= \
                    f"Residual AHI={ahi:.1f} on current therapy — recommend SpO2 enrollment " \
                    "(no wearable data on file yet)"

            if macro >= 2 and (isi >= 8 or z_natural > 0.30):
                requests.append("HRV monitoring (Hexoskin)")
                reasons["HRV monitoring (Hexoskin)"]= \
                    f"Declining adherence + elevated insomnia/dropout risk (ISI={isi:.0f}) — " \
                    "HRV enrollment could reveal autonomic stress driving disengagement"

            if ess >= 12 or (macro >= 1 and ess >= 8):
                requests.append("Sleep staging (SomnoArt)")
                reasons["Sleep staging (SomnoArt)"]= \
                    f"ESS={ess:.0f} (elevated sleepiness) despite CPAP use — sleep-staging " \
                    "enrollment could reveal residual sleep fragmentation"

            if z_risk >= 0.55 or bdi >= 20:
                requests.append("Blood pressure (BPM)")
                reasons["Blood pressure (BPM)"] = \
                    f"High overall dropout risk (z_risk={z_risk:.2f}) — cardiometabolic " \
                    "screening enrollment recommended alongside outreach"

        return requests[:3], reasons

    def recommend(self, row):
        intv, intv_reason= self.recommend_intervention(row)
        surveys, survey_reasons= self.recommend_surveys(row)
        biomarkers, biomarker_reasons= self.recommend_biomarker(row)

        return {
            "AtHomePatientId":row.get("AtHomePatientId"),
            "z_risk": round(self._f(row, "z_risk", 0), 3),
            "risk_level":self._s(row, "risk_level", "low"),
            "dropout_mechanism":self._s(row, "dropout_mechanism", "none"),
            "intervention_rec": intv,
            "intervention_reason":intv_reason,
            "surveys_to_send":"|".join(surveys) if surveys else "none",
            "survey_reasons": " ; ".join(f"{k}: {v}" for k,v in survey_reasons.items()),
            "request_biomarker": len(biomarkers) > 0,
            "biomarker_devices": "|".join(biomarkers) if biomarkers else "none",
            "biomarker_reasons": " ; ".join(f"{k}: {v}" for k,v in biomarker_reasons.items()),}

    def run(self, features_df, layer3_df):
        L3_COLS= ["z_natural","z_effectful","z_risk","z_proxy","risk_level",
                   "dropout_mechanism","h_days","hazard_rate","cpap_macro_state","combined_state","cpap_macro_name"]
        features_df= features_df.drop(
            columns=[c for c in L3_COLS if c in features_df.columns])
        merged= features_df.merge(
            layer3_df[["AtHomePatientId","z_natural","z_effectful","z_risk",
                        "z_proxy","risk_level","dropout_mechanism",
                        "h_days","hazard_rate","cpap_macro_state"]],on="AtHomePatientId", how="left",)
        merged[["z_risk","z_natural","z_effectful","h_days"]]= \
            merged[["z_risk","z_natural","z_effectful","h_days"]].fillna(0)
        merged["risk_level"] = merged["risk_level"].fillna("low")
        merged["dropout_mechanism"] = merged["dropout_mechanism"].fillna("none")

        results= [self.recommend(row) for _, row in merged.iterrows()]
        return pd.DataFrame(results)

#EVENT DETECTION + VIDEO TRIGGER

VIDEO_TOPICS = {
    "mask_comfort_and_seal": {
        "title":    "How to Achieve a Comfortable Mask Fit",
        "triggers": ["high_leaks", "usage_drop", "instability"],
        "addresses":"Mask leaks are reducing therapy effectiveness — patient likely uncomfortable",
        "signals":  "leaks_high_7d >= 0.40 (40%+ of nights with high leak) AND use_slope < 0",},
    "daily_routine_integration": {
        "title":    "Making CPAP Part of Your Nightly Routine",
        "triggers": ["usage_drop", "instability", "low_use_streak"],
        "addresses":"Irregular usage pattern — patient not consistently using CPAP",
        "signals":  "instability_idx > 0.5 AND low_use_streak ≥ 2",},
    "therapy_benefits_reinforcement": {
        "title":    "Why Every Night of CPAP Matters for Your Health",
        "triggers": ["natural_dropout_risk", "ess_rising", "declining_slope"],  
        "addresses":"Patient questioning therapy value — reinforce long-term benefits",
        "signals":  "z_natural > 0.30 OR (ESS_score > 8 AND use_slope < -0.2)",},
    "understanding_your_ahi": {
        "title":    "Understanding Your AHI Score and What It Means",
        "triggers": ["residual_ahi", "declining_adherence"],  
        "addresses":"AHI elevated despite CPAP — patient needs to understand pressure adjustment",
        "signals":  "ahi_mean_7d > 10 AND cpap_macro_state ≥ 2",},
    "managing_side_effects": {
        "title":    "Common CPAP Side Effects and How to Fix Them",
        "triggers": ["natural_dropout_risk", "non_responder", "low_response_rate"],  
        "addresses":"Intolerance patterns — patient likely experiencing side effects",
        "signals":  "z_natural > 0.40 AND response_rate < 0.25",},
    "sleep_hygiene_with_cpap": {
        "title":    "Optimising Your Sleep Environment with CPAP",
        "triggers": ["poor_sleep_eff", "high_wakeups", "high_psqi"],  
        "addresses":"Poor sleep quality despite CPAP — sleep hygiene issues compounding therapy",
        "signals":  "sleep_efficiency_7d < 0.70 OR wakeup_count_7d > 3",},
    "oxygen_and_heart_health": {
        "title":    "How CPAP Protects Your Heart and Oxygen Levels",
        "triggers": ["low_spo2", "afib", "high_bp"], 
        "addresses":"Cardiovascular risk signal — motivate therapy through heart health",
        "signals":  "spo2_7d < 90 OR afib_detected = 1 OR systolic_bp_7d > 150",},
    "mood_fatigue_and_cpap": {
        "title":    "CPAP and Its Impact on Mood, Energy and Mental Health",
        "triggers": ["high_bdi", "high_fss", "natural_dropout_risk"],
        "addresses":"Depression/fatigue signals — connect CPAP to mood improvement",
        "signals":  "BDI_score ≥ 11 OR FSS_score ≥ 4",},
    "what_happens_when_you_stop": {
        "title":    "What Happens to Your Health If You Stop CPAP",
        "triggers": ["critical_dropout_risk", "non_responder"],  
        "addresses":"High dropout probability — last-resort motivation before escalation",
        "signals":  "z_risk ≥ 0.60 AND h_days ≤ 60",},}

class EventDetector:
    """
    Detects multi-signal instability events and assigns video topics.
    """

    # Signal anomaly definitions and weights
    SIGNAL_CHECKS = {
        "usage_drop":{"col": "use_zscore", "op": "<", "threshold": -1.5, "w": 0.15},
        "declining_slope":{"col": "use_slope","op": "<",  "threshold": -0.3, "w": 0.10},
        "low_use_streak":{"col": "low_use_streak","op": ">=", "threshold":  2, "w": 0.08},
        "instability":{"col": "instability_idx","op": ">=", "threshold":  0.5, "w": 0.07},
        "high_leaks":{"col": "leaks_high_7d","op": ">=", "threshold": 0.40,"w": 0.08},
        "residual_ahi":{"col": "ahi_mean_7d","op": ">=", "threshold": 10,"w": 0.07},

        "ess_rising":{"col": "ESS_score","op": ">=", "threshold":8, "w": 0.12},
        "high_bdi":{"col": "BDI_score","op": ">=", "threshold": 11, "w": 0.12},
        "high_isi":{"col": "ISI_score","op": ">=", "threshold":  8, "w": 0.08},
        "high_psqi":{"col": "PSQI_score","op": ">=", "threshold":  6,"w": 0.08},
        "low_spo2":{"col": "spo2_7d","op": "<", "threshold": 90, "w": 0.10},
        "low_hrv":{"col": "hrv_rmssd_7d","op": "<", "threshold": 15,"w": 0.07},
        "poor_sleep_eff":  {"col": "sleep_efficiency_7d",  "op": "<",  "threshold":  0.70,"w": 0.08},
        "afib":{"col": "afib_detected","op": ">=", "threshold":  1,"w": 0.10},
        "high_bp":{"col": "systolic_bp_7d","op": ">", "threshold": 150,  "w": 0.05},
        "high_wakeups":{"col": "wakeup_count_7d","op": ">",  "threshold": 3,"w": 0.05},
        "natural_dropout_risk":{"col": "z_natural","op": ">=", "threshold": 0.30, "w": 0.10},
        "critical_dropout_risk": {"col": "z_risk","op": ">=", "threshold": 0.60, "w": 0.15},
        "low_response_rate": {"col": "response_rate","op": "<","threshold": 0.25, "w": 0.05},
        "high_fss":{"col": "FSS_score","op": ">=", "threshold": 4, "w": 0.05},
        "declining_adherence":{"col": "cpap_macro_state","op": ">=","threshold": 2,"w": 0.05},}

    def _check_signal(self, val, op, threshold):
        """Evaluate a single signal anomaly condition."""
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return False
        if op== "<":  return val < threshold
        if op== ">=": return val >= threshold
        if op== ">":  return val > threshold
        if op== "<=": return val <= threshold
        return False

    def _compute_event_score(self, row):
        """Compute weighted event score and list active signals."""
        active= []
        score= 0.0

        for signal, cfg in self.SIGNAL_CHECKS.items():
            val= row.get(cfg["col"])
            if val is not None:
                val = float(val) if not isinstance(val, float) else val
            if self._check_signal(val, cfg["op"], cfg["threshold"]):
                active.append(signal)
                score += cfg["w"]

        return min(score, 1.0), active

    def _assign_video_topic(self, active_signals, row):
        """
        Match active signals to video topics.
        Returns (topic_key, topic_title, reason).
        """
        topic_scores= {}
        for topic_key, topic_cfg in VIDEO_TOPICS.items():
            overlap= len(set(topic_cfg["triggers"]) & set(active_signals))
            if overlap > 0:
                topic_scores[topic_key] = overlap

        if not topic_scores:
            z= float(row.get("z_risk", 0))
            if z >= 0.60:
                best= "what_happens_when_you_stop"
            else:
                best= "daily_routine_integration"
        else:
            best= max(topic_scores, key=topic_scores.get)

        cfg= VIDEO_TOPICS[best]
        return best, cfg["title"], cfg["addresses"], cfg["signals"]

    def detect(self, row, non_responder=False):
        """
        Detect event and assign video for one patient.
        non_responder: True if patient had ≥2 failed interventions.
        """
        event_score, active= self._compute_event_score(row)

        if non_responder:
            active= active + ["non_responder"]

        # Video trigger thresholds
        trigger_threshold= 0.40 if non_responder else 0.60
        event_detected= event_score >= trigger_threshold

        if not event_detected:
            return {
                "AtHomePatientId":row.get("AtHomePatientId"),
                "event_detected":False,
                "event_score":round(event_score, 3),
                "n_active_signals": len(active),
                "active_signals":"|".join(active) if active else "none",
                "non_responder":non_responder,
                "video_topic":"none",
                "video_title":"none",
                "video_reason":"none",
                "video_trigger_signals":"none",}

        topic_key, title, reason, trigger_signals = self._assign_video_topic(active, row)

        return {
            "AtHomePatientId":row.get("AtHomePatientId"),
            "event_detected":True,
            "event_score":round(event_score, 3),
            "n_active_signals":len(active),
            "active_signals":"|".join(active),
            "non_responder":non_responder,
            "video_topic":topic_key,
            "video_title":title,
            "video_reason":reason,
            "video_trigger_signals": trigger_signals,}

    def run(self, features_df, layer3_df, triplets_df=None):
        """Run event detection for all patients."""
        non_resp_set= set()
        if triplets_df is not None:
            failed= (
                triplets_df.groupby("AtHomePatientId")["delta_use"]
                .apply(lambda x: (x < -0.5).sum()))
            non_resp_set = set(failed[failed >= 2].index)
        merged= features_df.merge(
            layer3_df[["AtHomePatientId", "z_risk", "z_natural",
                        "dropout_mechanism", "h_days", "cpap_macro_state"]],on="AtHomePatientId", how="left",)
        merged[["z_risk", "z_natural", "h_days", "cpap_macro_state"]]= merged[
            ["z_risk", "z_natural", "h_days", "cpap_macro_state"]].fillna(0)
        merged["dropout_mechanism"]= merged["dropout_mechanism"].fillna("none")

        results= []
        for _, row in merged.iterrows():
            pid= row.get("AtHomePatientId")
            non_responder= int(pid) in non_resp_set if pid else False
            results.append(self.detect(row, non_responder))

        return pd.DataFrame(results)

In [65]:
# RUN LAYERS 5 & 6  — self-contained, collision-safe
if "features" in dir() and isinstance(features, pd.DataFrame):
    _features= features.copy()
elif "features_with_evidence" in dir() and isinstance(features_with_evidence, pd.DataFrame):
    _features= features_with_evidence.copy()
else:
    _features= pd.read_csv("features_with_evidence.csv")
    print("Loaded features from features_with_evidence.csv")

if "layer3_results" in dir() and isinstance(layer3_results, pd.DataFrame):
    _l3 = layer3_results
elif "l3_results" in dir() and isinstance(l3_results, pd.DataFrame):
    _l3= l3_results
else:
    _l3= pd.read_csv("layer3_results.csv")
    print("Loaded layer3_results from layer3_results.csv")

_triplets= None
for name in ["triplets_enriched", "triplets"]:
    if name in dir() and isinstance(eval(name), pd.DataFrame):
        _triplets= eval(name)
        break
if _triplets is None:
    try:
        _triplets= pd.read_csv("layer4_triplets.csv")
        print("Loaded triplets from layer4_triplets.csv")
    except FileNotFoundError:
        pass

L3_COLS=["z_natural","z_effectful","z_risk","z_proxy","risk_level",
           "dropout_mechanism","h_days","hazard_rate","cpap_macro_state",
           "combined_state","cpap_macro_name"]
_features.drop(columns=[c for c in L3_COLS if c in _features.columns], inplace=True)

print(f"Ready: features={len(_features):,} |  l3={len(_l3):,} |  "
      f"triplets={len(_triplets):,}" if _triplets is not None else
      f"Ready: features={len(_features):,} |  l3={len(_l3):,} |  triplets=None")

print("\nRunning Layer 5 — Intervention & Data Recommender...")
l5_recommender= InterventionDataRecommender()
l5_results= l5_recommender.run(_features, _l3)
l5_results.to_csv("layer5_recommendations.csv", index=False)
print(f"Saved: layer5_recommendations.csv| {len(l5_results):,} patients")

print(f"\nIntervention distribution:")
print(l5_results["intervention_rec"].value_counts().to_string())
print(f"\nSurveys recommended (not 'none'):")
print(l5_results[l5_results["surveys_to_send"]!="none"]["surveys_to_send"].value_counts().head(10).to_string())
print(f"\nBiomarker monitoring requested: {l5_results['request_biomarker'].sum():,} patients")

print("\n\nRunning Layer 6 — Event Detection & Video Trigger...")
l6_detector= EventDetector()
l6_results= l6_detector.run(_features, _l3, _triplets)
l6_results.to_csv("layer6_events.csv", index=False)
print(f"Saved: layer6_events.csv | {len(l6_results):,} patients")

events= l6_results[l6_results["event_detected"]]
print(f"\nEvents detected: {len(events):,} ({len(events)/len(l6_results)*100:.1f}%)")
print(f"Non-responders: {l6_results['non_responder'].sum():,}")
if len(events) > 0:
    print(f"\nVideo topics assigned:")
    print(events["video_topic"].value_counts().to_string())
    print(f"\nTop active signals:")
    print(events["active_signals"].value_counts().head(8).to_string())

print("\n\nBuilding combined patient action plan...")
action_plan= l5_results.merge(
    l6_results[["AtHomePatientId","event_detected","event_score","video_topic","video_title","video_reason","non_responder"]],
    on="AtHomePatientId", how="left",)

video_mask= action_plan["event_detected"] == True
action_plan.loc[video_mask, "intervention_rec"] = "Video"
action_plan.loc[video_mask, "intervention_reason"] = (
    "Event detected — video: "
    + action_plan.loc[video_mask, "video_title"].fillna(""))

action_plan.to_csv("patient_action_plan.csv", index=False)
print(f"Saved: patient_action_plan.csv | {len(action_plan):,} patients")
print(f"\nFinal action distribution:")
print(action_plan["intervention_rec"].value_counts().to_string())

sample_cols= ["AtHomePatientId","z_risk","risk_level","dropout_mechanism","intervention_rec","surveys_to_send","event_detected","video_topic"]
sample_cols= [c for c in sample_cols if c in action_plan.columns]
at_risk = action_plan[action_plan["z_risk"] >= 0.30].head(5)
if len(at_risk) > 0:
    print(f"\nSample — {len(at_risk)} at-risk patients:")
    print(at_risk[sample_cols].to_string(index=False))

Ready: features=41,115 |  l3=41,115 |  triplets=30,182

Running Layer 5 — Intervention & Data Recommender...
Saved: layer5_recommendations.csv| 41,115 patients

Intervention distribution:
intervention_rec
Call     23016
SMS      16532
Video      824
Visit      743

Surveys recommended (not 'none'):
surveys_to_send
ESS             19170
ESS|PSQI        10919
ESS|BDI|ISI      5826
ESS|ISI|PSQI     3753
ESS|ISI|FSS       989
BDI|ISI           185
ISI                56
ESS|ISI            35
BDI                33
ISI|PSQI            7

Biomarker monitoring requested: 1,609 patients


Running Layer 6 — Event Detection & Video Trigger...
Saved: layer6_events.csv | 41,115 patients

Events detected: 254 (0.6%)
Non-responders: 1,270

Video topics assigned:
video_topic
sleep_hygiene_with_cpap           153
mask_comfort_and_seal              54
daily_routine_integration          34
therapy_benefits_reinforcement      5
mood_fatigue_and_cpap               5
understanding_your_ahi              2
oxy